# 🧠 LLM Context Management and Dynamic System Prompts

---
**Notebook developed by** SzuLun Huang <szuh@berkeley.edu>  
**Under the guidance of** Eric Van Dusen <ericvd@berkeley.edu>  
**UC Berkeley, Data Science**

---

## 📖 The Story

**Prof. Eric** teaches Data 8 — UC Berkeley's intro data science course. This semester he has 300 students, each at a different skill level. He pulls Zoe aside after class:

> *"Zoe, I want to build an AI study assistant for my students. But here's the problem — a Data 8 beginner and a Data 100 student need completely different explanations for the same question. Can you make it adapt automatically?"*

Zoe says yes. She has no idea what she's gotten herself into.

**This notebook is her journey** — every Part is a new problem she runs into, and a new concept she learns to solve it. By the end, she has a production-ready design and Prof. Eric has his assistant.

---

## 🚀 How to Start

1. Click **Kernel** in the top menu
2. Select **Restart Kernel and Run All Cells**
3. Wait about **1–2 minutes** for the model to load ⏳
4. Then explore the interactive sections below! ✅

> ⚠️ You only need to do this **once** each time you open the notebook.


# 📦 Step 0: Setup — run once (~1-2 min)

The three cells below import packages, locate the model file, and load it into memory.
Run them in order once per session — everything else depends on them.

In [55]:
# ── All imports ───────────────────────────────────────────────────────
import os
import json
import warnings
import time
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from llama_cpp import Llama
import re

warnings.filterwarnings('ignore')

# ── Utility helpers (all heavy logic lives in utils.py) ──────────────
from utils import (
    build_system_message, build_course_prompt,
    count_tokens, estimated_wait,
    summarize_full_text, extract_key_entities, compress_semantic,
    detect_profile_changes,
    ChatAssistant,
    visualize_context_window,
    retrieve, rag_chat,
    bubble, token_pill,
)

print('✅ Imports done.')


✅ Imports done.


In [56]:
# ── Model location ─────────────────────────────────────────────────
model_filename = 'Llama-3.2-1B-Instruct-Q4_K_M.gguf'
model_path     = f'/home/jovyan/shared/{model_filename}'

n_ctx     = 4096   # context window (tokens)
n_threads = 4      # CPU threads for inference

print(f'Model : {model_filename}')
print(f'Path  : {model_path}')


Model : Llama-3.2-1B-Instruct-Q4_K_M.gguf
Path  : /home/jovyan/shared/Llama-3.2-1B-Instruct-Q4_K_M.gguf


In [4]:
# ── Verify the model file exists before loading ─────────────────────
if os.path.exists(model_path):
    size_gb = os.path.getsize(model_path) / (1024**3)
    print(f'✅ Found  {model_filename}  ({size_gb:.2f} GB)')
else:
    print('❌ Model not found — ask your teacher to check the shared folder.')
    raise FileNotFoundError(f'Model not found at {model_path}')


✅ Found  Llama-3.2-1B-Instruct-Q4_K_M.gguf  (0.75 GB)


In [5]:
# ── Load model into memory (takes ~1-2 min on first run) ────────────
print('⏳ Loading model…  (this may take 1-2 minutes)\n')

model = Llama(
    model_path = model_path,
    n_ctx      = n_ctx,
    n_threads  = n_threads,
    verbose    = False,
)

clear_output(wait=True)
print('=' * 50)
print('✅ Model loaded and ready!')
print('=' * 50)
print(f'  Model   : {model_filename}')
print(f'  Context : {n_ctx} tokens')
print(f'  Threads : {n_threads}')


✅ Model loaded and ready!
  Model   : Llama-3.2-1B-Instruct-Q4_K_M.gguf
  Context : 4096 tokens
  Threads : 4


# 🧠 Part 1: "Why Doesn't It Remember Anything?"

Zoe's first prototype is just a few lines: she types a question, the AI answers. It works. She's excited.

She sends Prof. Eric a demo. He replies: *"Great! Can I ask a follow-up question?"*

She tries it — and the AI has no idea what was just discussed.

After some digging through the docs, she finds the answer: the model is completely **stateless**. Every API call starts fresh. It only knows what you pass in *right now*:

```python
response = model(messages)  # the model only sees what you pass in right now
```

If Zoe wants her assistant to remember the conversation, **she** has to keep track of it and pass it back every time. The `messages` list is the model's entire world.

## What goes in a messages list?

Each message has a `role` and `content`:

| Role | Who it's from | When to use |
|------|--------------|-------------|
| `system` | Zoe (as the developer) | Set behaviour, tone, or rules |
| `user` | The student using the assistant | What they typed |
| `assistant` | The AI | Previous AI responses |

> 💡 Right now Zoe is testing the assistant herself, so she's playing both roles. Later, Prof. Eric's students will be the `user`.

**👇 Work through the 3 challenges — you'll feel exactly what Zoe discovered.**


In [14]:
# ═══════════════════════════════════════════════════════════════════
#  Part 1 — Interactive Memory Explorer
# ═══════════════════════════════════════════════════════════════════

display(HTML("""
<style>
@keyframes fadeOut {
  from { opacity:1; transform:translateY(0);   max-height:120px; margin:5px 0; }
  to   { opacity:0; transform:translateY(-6px); max-height:0;    margin:0;     }
}
@keyframes fadeIn {
  from { opacity:0; transform:translateY(8px); }
  to   { opacity:1; transform:translateY(0);   }
}
@keyframes shake {
  0%,100%{transform:translateX(0)}
  20%{transform:translateX(-8px)} 40%{transform:translateX(8px)}
  60%{transform:translateX(-5px)} 80%{transform:translateX(5px)}
}
.bubble-wrap  { animation: fadeIn 0.35s ease both; }
.bubble-dying { animation: fadeOut 0.5s ease forwards; overflow:hidden; }
.amnesiac     { animation: shake 0.5s ease; }
.token-pill {
  display:inline-block; background:#313244; border-radius:20px;
  padding:2px 10px; font-size:0.76em; color:#a6adc8; margin-left:8px;
}
.token-pill.warn { background:#f9e2af22; color:#f9e2af; }
.token-pill.crit { background:#f38ba822; color:#f38ba8; }
</style>
"""))

# ── bubble renderer ──────────────────────────────────────────────────
def bubble(role, content, extra_class=""):
    cfg = {
        "system":    ("#f9e2af", "#f9e2af18", "left",  "⚙️ system"),
        "user":      ("#89b4fa", "#89b4fa18", "right", "👤 user"),
        "assistant": ("#a6e3a1", "#a6e3a118", "left",  "🤖 assistant"),
    }
    color, bg, side, label = cfg.get(role, ("#cdd6f4","#cdd6f418","left",role))
    align  = "flex-end"  if side == "right" else "flex-start"
    radius = "18px 18px 4px 18px" if side == "right" else "18px 18px 18px 4px"
    return f"""
    <div class="bubble-wrap {extra_class}"
         style="display:flex;justify-content:{align};margin:5px 0">
      <div style="max-width:75%">
        <div style="font-size:0.72em;color:{color};margin-bottom:3px;
                    text-align:{'right' if side=='right' else 'left'}">{label}</div>
        <div style="background:{bg};border:1px solid {color}44;
                    border-radius:{radius};padding:9px 14px;
                    color:#cdd6f4;font-size:0.87em;line-height:1.6">{content}</div>
      </div>
    </div>"""

def token_pill(msgs):
    try:
        n = sum(len(model.tokenize(m["content"].encode())) for m in msgs)
    except Exception:
        n = sum(len(m.get("content",""))//4 for m in msgs)
    cls = "crit" if n>500 else "warn" if n>150 else ""
    return f'<span class="token-pill {cls}">🪙 {n} tokens</span>'

# ── shared state ─────────────────────────────────────────────────────
state        = {"index": 0}
student_name = {"v": ""}
_c1_history  = {"msgs": []}

# ── FIXED system prompt ───────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are an AI tutor. A student is talking to you. "
    "The student's name and background will appear in their first message. "
    "IMPORTANT: When the student introduces themselves, acknowledge them by name in your reply. "
    "When asked who the student is, state their name and background exactly as they told you. "
    "For example: 'The student I am talking to is Zoey, a Data 100 student learning machine learning.' "
    "Never say you don't know the student's name if they have already introduced themselves. "
    "Keep replies to 1-2 sentences."
)
# ── widgets ──────────────────────────────────────────────────────────
banner_out  = widgets.Output()
preview_out = widgets.Output()
output_area = widgets.Output()

c1_name = widgets.Text(placeholder="e.g. your name",
                       layout=widgets.Layout(width="280px"))
c1_like = widgets.Text(placeholder="e.g. Data 8 student, just starting with the datascience library",
                       layout=widgets.Layout(width="480px"))
c1_form = widgets.VBox([
    widgets.HTML('<div style="color:#a6adc8;font-size:0.84em;margin:10px 0 4px 0">'
                 '👤 Your name:</div>'),
    c1_name,
    widgets.HTML('<div style="color:#a6adc8;font-size:0.84em;margin:8px 0 4px 0">'
                 '📚 Something about yourself (course, background, what you\'re learning):</div>'),
    c1_like,
], layout=widgets.Layout(display="none", padding="0 0 10px 0"))

send_btn = widgets.Button(description="▶ Send", button_style="primary",
                          layout=widgets.Layout(width="110px", margin="10px 6px 0 0"))
next_btn = widgets.Button(description="Next →", button_style="info",
                          layout=widgets.Layout(width="110px", margin="10px 0 0 0"))

CHALLENGES = [
    {
        "id": "c1", "label": "Challenge 1 of 2",
        "title": "The AI remembers — because YOU gave it the memory",
        "color": "#89b4fa", "icon": "🧠",
        "instruction": "Enter your name and something about yourself, then hit <strong>Send</strong>.",
        "hint": "Every message in the list is part of the AI's memory. It can see all of them.",
        "form": "c1",
    },
    {
        "id": "c2", "label": "Challenge 2 of 2",
        "title": "Watch your memory disappear.",
        "color": "#f38ba8", "icon": "💀",
        "instruction": "Watch the messages vanish — then hit <strong>Send</strong> to confirm the AI forgot everything.",
        "hint": "The AI only sees what's in the list right now. No list = no memory.",
        "form": None,
    },
]

# ── banner & layout ───────────────────────────────────────────────────
def load_challenge(idx):
    c = CHALLENGES[idx]
    with banner_out:
        clear_output(wait=True)
        display(HTML(f"""
        <div style="background:#1e1e2e;border:2px solid {c['color']};
                    border-radius:12px;padding:16px 20px;margin:10px 0">
          <div style="display:flex;align-items:center;gap:10px;margin-bottom:8px">
            <span style="font-size:1.6em">{c['icon']}</span>
            <div>
              <div style="color:{c['color']};font-weight:bold;font-size:0.85em">{c['label']}</div>
              <div style="color:#cdd6f4;font-weight:bold;font-size:1em">{c['title']}</div>
            </div>
          </div>
          <div style="color:#cdd6f4;font-size:0.9em;margin-bottom:6px">{c['instruction']}</div>
          <div style="color:#a6adc8;font-size:0.8em">💡 {c['hint']}</div>
        </div>"""))

    c1_form.layout.display = "" if c["form"] == "c1" else "none"

    if c["id"] == "c1":
        refresh_c1_preview()
    else:
        saved = _c1_history.get("msgs")
        render_preview(saved if saved else [], label="📨 This was your memory from Challenge 1…")

    next_btn.disabled    = (idx == len(CHALLENGES) - 1)
    next_btn.description = "✅ Done" if idx == len(CHALLENGES) - 1 else "Next →"
    with output_area:
        clear_output()

# ── preview ───────────────────────────────────────────────────────────
def render_preview(messages, label=None):
    label = label or f'📨 Messages the model will receive {token_pill(messages)}'
    bubs  = "".join(bubble(m["role"], m["content"]) for m in messages)
    with preview_out:
        clear_output(wait=True)
        display(HTML(
            f'<div style="background:#1e1e2e;border-radius:10px;padding:14px">'
            f'<div style="color:#585b70;font-size:0.78em;margin-bottom:6px">{label}</div>'
            f'{bubs}</div>'))

def refresh_c1_preview(*_):
    name = c1_name.value.strip() or "you"
    like = c1_like.value.strip() or "…"
    msgs = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Hi! My name is {name}. {like}."},
        {"role": "assistant", "content": "( AI will reply here after Turn 1 )"},
        {"role": "user",      "content": "What is my name, and what is my background?"},
    ]
    render_preview(msgs)

c1_name.observe(refresh_c1_preview, names="value")
c1_like.observe(refresh_c1_preview, names="value")

# ── amnesia animation ─────────────────────────────────────────────────
def play_amnesia_then_send():
    full_msgs = _c1_history.get("msgs", [])
    if not full_msgs:
        with output_area:
            clear_output()
            display(HTML('<p style="color:#f38ba8">⚠️ Please complete Challenge 1 first!</p>'))
        return

    for i in range(len(full_msgs) - 1):
        surviving = [
            bubble(m["role"], m["content"],
                   extra_class="bubble-dying" if j == i else "")
            for j, m in enumerate(full_msgs)
        ]
        with preview_out:
            clear_output(wait=True)
            display(HTML(
                '<div style="background:#1e1e2e;border-radius:10px;padding:14px">'
                '<div style="color:#f38ba8;font-size:0.78em;margin-bottom:6px">'
                '🗑️ Erasing memory…</div>'
                + "".join(surviving) + "</div>"))
        time.sleep(0.6)

    lone = [{"role": "user", "content": "What is the name of the student you are talking to, and what is their background?"}]
    with preview_out:
        clear_output(wait=True)
        display(HTML(
            '<div style="background:#1e1e2e;border-radius:10px;padding:14px">'
            f'<div style="color:#f38ba8;font-size:0.78em;margin-bottom:6px">'
            f'📨 All that remains {token_pill(lone)}</div>'
            + bubble(lone[0]["role"], lone[0]["content"]) + "</div>"))

    resp  = model.create_chat_completion(messages=lone, max_tokens=60, temperature=0.7)
    reply = resp["choices"][0]["message"]["content"].strip()
    name  = student_name["v"] or "you"

    with output_area:
        clear_output(wait=True)
        display(HTML(f"""
        <div class="amnesiac" style="background:#1e1e2e;border-radius:10px;
                    padding:14px;margin-top:8px">
          <div style="color:#585b70;font-size:0.78em;margin-bottom:8px">🤖 Model reply</div>
          {bubble("assistant", reply)}
          <div style="background:#f38ba822;border:2px solid #f38ba8;border-radius:10px;
               padding:12px 16px;margin-top:12px;text-align:center">
            <div style="font-size:1.4em;margin-bottom:4px">🫥</div>
            <div style="color:#f38ba8;font-weight:bold">Complete amnesia.</div>
            <div style="color:#a6adc8;font-size:0.82em;margin-top:6px;line-height:1.7">
              The model received <strong style="color:#f38ba8">1 message</strong> — no history, no name, nothing.<br>
              It has no idea who <strong style="color:#cdd6f4">{name}</strong> is.<br><br>
              <strong style="color:#cdd6f4">This is what Zoe's first broken prototype felt like.</strong><br>
              <span style="color:#585b70">The fix? Pass the full messages list every time. That's what Part 2 builds.</span>
            </div>
          </div>
        </div>"""))

# ── send logic ────────────────────────────────────────────────────────
def on_send(btn):
    c = CHALLENGES[state["index"]]
    send_btn.disabled    = True
    send_btn.description = "⏳ …"

    if c["id"] == "c2":
        with output_area:
            clear_output()
        play_amnesia_then_send()

    else:  # C1
        name = c1_name.value.strip()
        like = c1_like.value.strip()
        if not name or not like:
            with output_area:
                clear_output()
                display(HTML('<p style="color:#f38ba8">⚠️ Enter your name and something about yourself first!</p>'))
            send_btn.disabled    = False
            send_btn.description = "▶ Send"
            return
        student_name["v"] = name

        # Turn 1
        turn1 = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Hi! My name is {name}. {like}."},
        ]
        with output_area:
            clear_output()
            display(HTML('<p style="color:#89b4fa;font-size:0.88em">⏳ Turn 1: sending introduction…</p>'))
        resp1    = model.create_chat_completion(messages=turn1, max_tokens=60, temperature=0.7)
        ai_reply = resp1["choices"][0]["message"]["content"].strip()

        # Turn 2
        turn2 = turn1 + [
            {"role": "assistant", "content": ai_reply},
            {"role": "user",      "content": "What is the name of the student you are talking to, and what is their background?"},
        ]
        _c1_history["msgs"] = turn2

        with output_area:
            clear_output()
            display(HTML('<p style="color:#89b4fa;font-size:0.88em">⏳ Turn 2: asking follow-up…</p>'))
        resp2 = model.create_chat_completion(messages=turn2, max_tokens=80, temperature=0.7)
        final = resp2["choices"][0]["message"]["content"].strip()

        bubs = "".join(bubble(m["role"], m["content"]) for m in turn2)
        with output_area:
            clear_output(wait=True)
            display(HTML(f"""
            <div style="background:#1e1e2e;border-radius:10px;padding:14px;margin-top:8px">
              <div style="color:#585b70;font-size:0.78em;margin-bottom:8px">
                📨 What the model received {token_pill(turn2)}</div>
              {bubs}
              <div style="border-top:1px solid #313244;margin:10px 0"></div>
              <div style="color:#585b70;font-size:0.78em;margin-bottom:6px">🤖 Model reply</div>
              {bubble("assistant", final)}
              <div style="background:#89b4fa22;border:1px solid #89b4fa44;border-radius:8px;
                   padding:10px 14px;margin-top:10px;font-size:0.82em;color:#a6adc8">
                ✅ The AI knows your name because <strong style="color:#cdd6f4">you gave it the memory</strong>.
                Hit <strong style="color:#89b4fa">Next →</strong> to see what happens when that memory disappears.
              </div>
            </div>"""))

    send_btn.disabled    = False
    send_btn.description = "▶ Send"

def on_next(btn):
    nxt = state["index"] + 1
    if nxt < len(CHALLENGES):
        state["index"] = nxt
        load_challenge(nxt)

send_btn.on_click(on_send)
next_btn.on_click(on_next)

# ── initial render ────────────────────────────────────────────────────
display(widgets.HTML("""
<div style="background:#1e1e2e;padding:16px 20px;border-radius:12px;margin-bottom:4px">
  <h3 style="color:#cdd6f4;margin:0 0 6px 0">🧠 Part 1: The messages List is the AI's Entire Memory</h3>
  <p style="color:#a6adc8;margin:0;font-size:0.88em">
    Two challenges. Each one lets you <em>feel</em> how AI memory works — and breaks.
  </p>
</div>
"""))
load_challenge(0)
display(banner_out, c1_form, preview_out,
        widgets.HBox([send_btn, next_btn]), output_area)


HTML(value='\n<div style="background:#1e1e2e;padding:16px 20px;border-radius:12px;margin-bottom:4px">\n  <h3 s…

Output()

Output()

Output()


# 👤 Part 2: "300 Students, 300 Different Needs"

Zoe shows the working prototype to Prof. Eric. He's happy — but immediately has a new request:

> *"This is great. But my Data 8 students are complete beginners — they need simple language and analogies. My Data 100 students are much more advanced — they'd find that patronising. Can the same assistant handle both?"*

Zoe's first instinct: write two different system prompts. But Prof. Eric has 300 students. She can't hardcode a prompt for each one.

Her solution: **store the student's info as a dict, and let code write the prompt automatically.**

```python
profile = {"name": "Student", "expertise": "beginner", ...}
system_prompt = build_system_message(profile)  # prompt writes itself
```

Change one field in the profile → the whole prompt regenerates. One function, any student.



## 🎓 Demo: Data 8 vs Data 100 

Same question, two very different students:

| Course | Student type | What they need |
|--------|-------------|----------------|
| **Data 8** | Complete beginner | Simple language, analogies, `datascience` library |
| **Data 100** | Experienced | Concise, technical, `pandas` / industry tools |

**What to watch:** Ask the AI *"How do I read a CSV file?"* — but change the `course` field.

> 🔍 The model is identical. The question is identical. Only the system prompt changes — but the answer is completely different. This is exactly what Zoe builds for Prof. Eric.


In [15]:
# ── Data 8 vs Data 100 System Prompt Builder ─────────────────────────
question = "How do I read a CSV file?"

COURSE_COLORS = {
    "Data 8":   {"border": "#a6e3a1", "label": "#a6e3a1", "keyword": "datascience", "kw_color": "#a6e3a1"},
    "Data 100": {"border": "#89b4fa", "label": "#89b4fa", "keyword": "pandas",      "kw_color": "#89b4fa"},
}

def highlight(text, keyword, color):
    return text.replace(
        keyword,
        f'<span style="color:{color};font-weight:bold;background:#1e1e2e;'
        f'padding:1px 5px;border-radius:3px">{keyword}</span>'
    )

output_area = widgets.Output()

def run_comparison(_=None):
    run_btn.disabled = True
    run_btn.description = "⏳ Running..."
    with output_area:
        clear_output()
        display(HTML('<div style="color:#a6adc8;font-size:0.88em;padding:8px 0">⏳ Querying both courses...</div>'))

    results = {}
    prompts = {}
    for course, cfg in COURSE_COLORS.items():
        prompts[course] = build_course_prompt(course)
        resp = model.create_chat_completion(
            messages=[
                {"role": "system", "content": prompts[course]},
                {"role": "user",   "content": question},
            ],
            max_tokens=150,
            temperature=0.7,
        )
        results[course] = resp["choices"][0]["message"]["content"].strip()

    with output_area:
        clear_output()
        html_parts = ['<div style="display:flex;gap:14px;margin-top:10px">']
        for course, cfg in COURSE_COLORS.items():
            reply_html = highlight(results[course], cfg["keyword"], cfg["kw_color"])
            reply_html = reply_html.replace(
                "Table.read_table",
                f'<span style="color:{cfg["kw_color"]};font-weight:bold">Table.read_table</span>'
            ).replace(
                "pd.read_csv",
                f'<span style="color:{cfg["kw_color"]};font-weight:bold">pd.read_csv</span>'
            )

            html_parts.append(f"""
            <div style="flex:1;background:#1e1e2e;border:2px solid {cfg['border']};
                        border-radius:10px;padding:14px">

              <div style="color:{cfg['label']};font-weight:bold;font-size:1em;margin-bottom:10px">
                🎓 {course}
              </div>

              <!-- System prompt section -->
              <div style="color:#a6adc8;font-size:0.75em;font-weight:bold;
                          text-transform:uppercase;letter-spacing:0.05em;margin-bottom:4px">
                ⚙️ System Prompt
              </div>
              <div style="background:#181825;border:1px solid {cfg['border']}44;
                          border-radius:6px;padding:8px 10px;
                          color:#a6adc8;font-size:0.78em;line-height:1.6;
                          max-height:90px;overflow-y:auto;
                          white-space:pre-wrap;margin-bottom:10px">{prompts[course]}</div>

              <!-- Arrow -->
              <div style="text-align:center;color:#585b70;font-size:1.2em;margin-bottom:8px">↓</div>

              <!-- AI response section -->
              <div style="color:#a6adc8;font-size:0.75em;font-weight:bold;
                          text-transform:uppercase;letter-spacing:0.05em;margin-bottom:4px">
                🤖 AI Response
              </div>
              <div style="background:#313244;padding:10px;border-radius:6px;
                          color:#cdd6f4;font-size:0.87em;line-height:1.6;
                          white-space:pre-wrap">{reply_html}</div>

            </div>""")

        html_parts.append('</div>')
        html_parts.append("""
        <div style="margin-top:12px;background:#313244;padding:10px 14px;border-radius:8px;
                    font-size:0.85em;color:#a6adc8">
          ✨ <strong style="color:#f9e2af">Key observation:</strong>
          Same model, same question — only the system prompt changed.<br>
          The highlighted library name shows exactly where the behaviour diverged.
        </div>""")
        display(HTML("".join(html_parts)))

    run_btn.disabled = False
    run_btn.description = "▶ Run Again"

run_btn = widgets.Button(
    description="▶ Run Comparison",
    button_style="primary",
    layout=widgets.Layout(width="160px", margin="0 0 10px 0")
)
run_btn.on_click(run_comparison)

display(widgets.HTML("""
<div style="background:#1e1e2e;padding:14px 18px;border-radius:10px;margin-bottom:10px">
  <h4 style="color:#cdd6f4;margin:0 0 6px 0">🎓 Data 8 vs Data 100 — Side-by-Side</h4>
  <p style="color:#a6adc8;margin:0;font-size:0.88em">
    Question: <em>"How do I read a CSV file?"</em><br>
    Notice how the <strong style="color:#f9e2af">system prompt</strong> changes the library the AI recommends.
  </p>
</div>
"""), run_btn, output_area)

HTML(value='\n<div style="background:#1e1e2e;padding:14px 18px;border-radius:10px;margin-bottom:10px">\n  <h4 …

Button(button_style='primary', description='▶ Run Comparison', layout=Layout(margin='0 0 10px 0', width='160px…

Output()


## 🪙 Token Counter: The Hidden Cost Zoe Didn't Expect

The assistant is working. Prof. Eric is happy. But a week into the semester, Zoe notices something: **the assistant is getting slower.**

Every message — system prompt, chat history, new question — consumes **tokens** from the model's context window (4,096 tokens for our local model). More tokens in = longer wait.

| What gets sent | Approx. tokens |
|---------------|----------------|
| Simple question only | ~8 tokens |
| Question + system prompt | ~100 tokens |
| Question + system prompt + 20–30 turns of history | ~1,000–3,000 tokens |

Run the cell below to see the **exact numbers** for each scenario.

> 💡 This is why **history compression** (Part 3) is not optional — especially on a local 1B model where every extra token adds real waiting time for students.


In [16]:
# ── Token Counter Demonstration ──────────────────────────────────────
simple_question  = "How do I read a CSV file in Python?"
data100_prompt   = build_course_prompt("Data 100")

msgs_simple = [
    {"role": "user", "content": simple_question}
]
msgs_with_prompt = [
    {"role": "system", "content": data100_prompt},
    {"role": "user",   "content": simple_question},
]
msgs_with_history = [
    {"role": "system", "content": data100_prompt},
    {"role": "user",      "content": "Hi! I'm Zoe, a student from Taiwan studying AI at UC Berkeley."},
    {"role": "assistant", "content": "Welcome Zoe! That's exciting. What are you working on?"},
    {"role": "user",      "content": "I'm learning about context management in LLMs for a class project."},
    {"role": "assistant", "content": "Great topic! Context management is key for building real AI apps."},
    {"role": "user",      "content": "I'm also learning pandas and scikit-learn for the data side."},
    {"role": "assistant", "content": "Nice combo. Are you using Jupyter or a local environment?"},
    {"role": "user",      "content": simple_question},
]

t1 = count_tokens(msgs_simple)
t2 = count_tokens(msgs_with_prompt)
t3 = count_tokens(msgs_with_history)
SPEED = 25

rows = [
    ("Simple question only",     t1),
    ("+ System prompt",          t2),
    ("+ 6-turn history preview", t3),
]

html = ['<div style="background:#1e1e2e;border-radius:10px;padding:16px 20px;margin-top:8px">']
html.append('<h4 style="color:#cdd6f4;margin:0 0 12px 0">🪙 Token Count & Estimated Wait Time</h4>')
html.append('<table style="width:100%;border-collapse:collapse;font-size:0.88em">')
html.append('''<tr style="background:#2a2a3d;border-bottom:1px solid #45475a">
  <th style="text-align:left;padding:6px 10px;color:#cdd6f4">Scenario</th>
  <th style="text-align:right;padding:6px 10px;color:#cdd6f4">Tokens</th>
  <th style="text-align:right;padding:6px 10px;color:#cdd6f4">% of 4096</th>
  <th style="text-align:right;padding:6px 10px;color:#cdd6f4">Est. wait (@25 tok/s)</th>
</tr>''')

bar_colors = ["#a6e3a1", "#89b4fa", "#f38ba8"]
for (label, t), color in zip(rows, bar_colors):
    pct   = t / 4096 * 100
    wait  = estimated_wait(t, SPEED)
    bar_w = max(4, int(pct * 1.5))
    html.append(f'''<tr style="background:#1e1e2e;border-bottom:1px solid #313244">
  <td style="padding:8px 10px;color:#cdd6f4">{label}</td>
  <td style="text-align:right;padding:8px 10px;color:{color};font-weight:bold">{t}</td>
  <td style="text-align:right;padding:8px 10px">
    <span style="display:inline-block;width:{bar_w}px;height:10px;
                 background:{color};border-radius:3px;vertical-align:middle"></span>
    <span style="color:#cdd6f4;margin-left:6px">{pct:.1f}%</span>
  </td>
  <td style="text-align:right;padding:8px 10px;color:#cdd6f4">~{wait:.1f} s</td>
</tr>''')

html.append('</table>')
html.append(f'''<div style="margin-top:12px;padding:10px 14px;background:#313244;border-radius:8px;
                            font-size:0.85em;line-height:1.8">
  💡 <strong style="color:#f9e2af">Cost us time:</strong>
  <span style="color:#cdd6f4"> Adding a system prompt multiplies token count by ~{t2/t1:.0f}×.</span><br>
  <span style="color:#cdd6f4">Adding 6 turns of history multiplies it by ~{t3/t1:.0f}× vs. the bare question.</span><br>
  <span style="color:#cdd6f4">With 25 full turns (Zoe\'s story below), expect </span>
  <strong style="color:#f38ba8">1,000 + tokens</strong>
  <span style="color:#cdd6f4"> and ~{1000/SPEED:.0f}+ seconds of wait time on this local 1B model.</span><br>
  👉 <strong style="color:#cdd6f4">This is exactly why history compression (Part 3) matters.</strong>
</div>
</div>''')

display(HTML("".join(html)))

Scenario,Tokens,% of 4096,Est. wait (@25 tok/s)
Simple question only,8,0.2%,~0.3 s
+ System prompt,85,2.1%,~3.4 s
+ 6-turn history preview,174,4.2%,~7.0 s



## 📖 Part 2b: How the Assistant Learns Who Zoe Is

Prof. Eric asks: *"Can the assistant learn a student's background just from conversation — without making them fill out a form?"*

Yes. Instead of asking *"What's your skill level?"* upfront, the assistant can **infer the profile from what the student says** across multiple turns.

Below is a **simulated 25-turn conversation** with Zoe herself as the student. Her background — Taiwan, AI focus, pandas experience — emerges naturally through the chat.

**What to observe:**
- Early turns: Zoe is just asking questions, profile is mostly empty
- Middle turns: the AI starts tailoring responses to her background
- Later turns: the AI knows her skills, goals, and style without ever being told explicitly

Run the cell and watch how the system prompt grows with the history.


In [17]:
# ── DATASET: Zoe's Storyline History (25 turns) ──────────────────────
ZOE_HISTORY = [
    {"role": "user",      "content": "Hi! I'm Zoe. I'm from Taiwan and I'm studying at UC Berkeley."},
    {"role": "assistant", "content": "Welcome Zoe! Great to meet you. What are you studying?"},
    {"role": "user",      "content": "I'm focusing on AI and machine learning. It's my first semester here."},
    {"role": "assistant", "content": "That's exciting! Berkeley has a great CS program. What draws you to AI?"},
    {"role": "user",      "content": "I want to understand how language models work — especially memory and context."},
    {"role": "assistant", "content": "Context management is a fascinating area. Are you working on a project?"},
    {"role": "user",      "content": "Yes, I'm building a teaching notebook about LLM context management for a class."},
    {"role": "assistant", "content": "That sounds like a great project. What tools are you using?"},
    {"role": "user",      "content": "Python, Jupyter, and llama-cpp-python with a local Llama model."},
    {"role": "assistant", "content": "Nice setup! Running locally avoids API costs. How's the performance?"},
    {"role": "user",      "content": "It's a bit slow — the 1B model takes a few seconds per response."},
    {"role": "assistant", "content": "That's expected. Reducing token count helps. Have you tried history compression?"},
    {"role": "user",      "content": "Not yet. That's actually one of the topics I want to teach in the notebook."},
    {"role": "assistant", "content": "Perfect timing then. Summarization and entity extraction are two common approaches."},
    {"role": "user",      "content": "I also want to show students the difference between Data 8 and Data 100 workflows."},
    {"role": "assistant", "content": "Good idea. The datascience vs pandas distinction is a classic Berkeley contrast."},
    {"role": "user",      "content": "Exactly! I think system prompts are the key to making that demo work."},
    {"role": "assistant", "content": "Right — change one field in the profile, the whole prompt regenerates."},
    {"role": "user",      "content": "I'm also learning about token budgets. Every token = more wait time locally."},
    {"role": "assistant", "content": "That's a great insight to teach. Students often ignore token costs until they feel it."},
    {"role": "user",      "content": "My background is more stats than CS, so I'm still getting comfortable with Python."},
    {"role": "assistant", "content": "Stats is a great foundation. pandas and numpy will feel natural to you."},
    {"role": "user",      "content": "I also want a fun mode where the AI speaks in a playful, encouraging tone."},
    {"role": "assistant", "content": "Easy — just add that as a style option in the system prompt."},
    {"role": "user",      "content": "Cool, I think that will make the notebook more fun. Thanks for helping me plan this!"},
]

from IPython.display import display, HTML

def display_dataset_professional(history):
    def count_tokens(text):
        return max(1, len(text) // 4)
    total_tok = count_tokens(" ".join(m["content"] for m in history))

    rows = ""
    for m in history:
        is_user    = m["role"] == "user"
        role_color = "#1a5fb4" if is_user else "#26a269"
        label      = m["role"].upper()
        rows += f"""
        <tr style="border-bottom:1px solid #eee">
          <td style="padding:12px 14px; vertical-align:top; width:110px;
                     font-weight:700; color:{role_color}; font-size:0.88em; text-align:left;">{label}</td>
          <td style="padding:12px 14px; color:#000000; line-height:1.6; font-size:0.9em; text-align:left;">{m['content']}</td>
        </tr>"""

    html = f"""
    <div style="font-family: sans-serif; margin:10px 0">
      <div style="background:#1e2d40; border:1px solid #3b5268; border-radius:10px 10px 0 0;
                  padding:12px 16px; position:relative;">
        <div style="color:#f0f6ff; font-weight:700; font-size:1em; text-align:center;">
          📖 Full Conversation History — Zoe
        </div>
        <div style="position:absolute; right:16px; top:14px; font-size:0.75em; color:#7ea8c9; display:flex; gap:10px;">
          <span>{len(history)} turns</span>
          <span style="color:#3b5268">|</span>
          <span style="color:#38bdf8; font-weight:700">{total_tok} tokens</span>
        </div>
      </div>

      <div style="max-height:300px; overflow-y:auto;
                  border:1px solid #3b5268; border-top:none;
                  border-radius:0 0 10px 10px; background:#ffffff">
        <table style="width:100%; border-collapse:collapse; table-layout:fixed;">
          <thead>
            <tr style="background:#f8f9fa; border-bottom:2px solid #dee2e6; position:sticky; top:0; z-index:10;">
              <th style="padding:10px 14px; width:110px; text-align:center;
                         color:#444444; font-size:0.75em; text-transform:uppercase;
                         letter-spacing:0.1em; font-weight:bold;">Role</th>
              <th style="padding:10px 14px; text-align:center;
                         color:#444444; font-size:0.75em; text-transform:uppercase;
                         letter-spacing:0.1em; font-weight:bold;">Message</th>
            </tr>
          </thead>
          <tbody>{rows}</tbody>
        </table>
      </div>
    </div>"""

    display(HTML(html))

# Execute to show the result
display_dataset_professional(ZOE_HISTORY)

Role,Message
USER,Hi! I'm Zoe. I'm from Taiwan and I'm studying at UC Berkeley.
ASSISTANT,Welcome Zoe! Great to meet you. What are you studying?
USER,I'm focusing on AI and machine learning. It's my first semester here.
ASSISTANT,That's exciting! Berkeley has a great CS program. What draws you to AI?
USER,I want to understand how language models work — especially memory and context.
ASSISTANT,Context management is a fascinating area. Are you working on a project?
USER,"Yes, I'm building a teaching notebook about LLM context management for a class."
ASSISTANT,That sounds like a great project. What tools are you using?
USER,"Python, Jupyter, and llama-cpp-python with a local Llama model."
ASSISTANT,Nice setup! Running locally avoids API costs. How's the performance?


In [18]:
# ── Setup ─────────────────────────────────────────────────────────────
simple_question = "Based on my background and what I'm working on, what should I focus on to get better at AI?"

msgs_full_zoe = [
    {
        "role": "system",
        "content": (
            "You are a specialized AI mentor for Data Science students at UC Berkeley. "
            "The user is Zoe. You MUST tailor your advice to her specific background: "
            "Taiwanese, Statistics background, UC Berkeley student, building an LLM context notebook. "
            "FORMAT REQUIREMENT: Provide 5 specific focus areas. Each point must start with a "
            "BOLD KEYWORD followed by a very brief one-sentence explanation. "
            "Example format: '**Keyword**: Brief explanation.' "
            "Do NOT give generic advice. Keep the bolded parts strictly to the keywords."
        )
    },
    *ZOE_HISTORY,
    {"role": "user", "content": simple_question},
]

# ── Bold fix: replace **text** with proper <b>text</b> ───────────────
def render_bold(text):
    # fix **keyword**: replaces pairs correctly using regex
    text = re.sub(r'\*\*(.+?)\*\*', r'<strong style="color:#f0f6ff">\1</strong>', text)
    text = text.replace("\n", "<br>")
    return text

# ── 1. Input panel ────────────────────────────────────────────────────
display(HTML(f"""
<div style="font-family:'IBM Plex Mono','Fira Code',monospace;
            background:#0d1420;border:1px solid #3b5268;
            border-radius:12px;padding:18px 20px;margin:10px 0;color:#e2e8f0">

  <div style="font-size:0.63em;color:#7ea8c9;text-transform:uppercase;
              letter-spacing:0.15em;margin-bottom:8px">Input Context</div>
  <h3 style="color:#f0f6ff;margin:0 0 14px;font-size:1.0em">🔍 Student Profile</h3>

  <div style="display:flex;flex-direction:column;gap:6px;
              font-size:0.8em;color:#94b8d4;line-height:1.7;margin-bottom:14px">
    <div><span style="color:#475569">Name / Origin  </span> Zoe (Taiwan)</div>
    <div><span style="color:#475569">Affiliation    </span> UC Berkeley — Statistics background</div>
    <div><span style="color:#475569">Current project</span> LLM Context Management Notebook</div>
  </div>

  <div style="background:#020408;border:1px solid #38bdf844;border-left:3px solid #38bdf8;
              border-radius:0 8px 8px 0;padding:12px 14px;font-size:0.82em;
              color:#cbd5e1;line-height:1.6;font-style:italic">
    "{simple_question}"
  </div>

</div>
"""))

# ── 2. Inference ──────────────────────────────────────────────────────
print("⏳ Generating personalized roadmap...")
try:
    resp = model.create_chat_completion(
        messages=msgs_full_zoe,
        max_tokens=250,
        temperature=0.4,
    )
    final_reply = resp["choices"][0]["message"]["content"].strip()
except Exception as e:
    final_reply = f"Error: {e}"

# ── 3. Output panel ───────────────────────────────────────────────────
display(HTML(f"""
<div style="font-family:'IBM Plex Mono','Fira Code',monospace;
            background:#0d1420;border:2px solid #34d399;
            border-radius:12px;padding:18px 20px;margin-top:10px;color:#e2e8f0">

  <div style="font-size:0.63em;color:#34d399;text-transform:uppercase;
              letter-spacing:0.15em;margin-bottom:8px">Output</div>
  <h3 style="color:#f0f6ff;margin:0 0 14px;font-size:1.0em">🎯 Personalized AI Learning Roadmap</h3>

  <div style="background:#020408;border:1px solid #34d39922;border-radius:8px;
              padding:14px 16px;font-size:0.82em;color:#cbd5e1;line-height:1.9">
    {render_bold(final_reply)}
  </div>

  <div style="margin-top:12px;font-size:0.72em;color:#334155;font-style:italic">
    Focus areas are prioritized based on Zoe's statistical background and local LLM development context.
  </div>

</div>
"""))

⏳ Generating personalized roadmap...


## 🎨 Bonus: Change the AI's Style with One Line

System prompts don't just control **content** — they control **tone and style** too.  
Here we swap a single instruction to show how the same answer sounds completely different depending on who's "speaking".

> 💡 **What to notice:** Same question, same model, same facts — only the style instruction changes.


In [19]:
# ── Style options ─────────────────────────────────────────────────────
STYLES = {
    "👤 Friendly Senior":   "You are a helpful senior student. Answer in 1-2 sentences only, like you're explaining to a friend.",
    "🎉 Encouraging Coach": "Respond in an upbeat tone. Answer in 1-2 sentences only, then add one short encouragement.",
    "🎓 Professor":         "Respond formally. Answer in 1-2 sentences only, using precise terminology.",
    "🎤 Rap / Freestyle":   "Respond ONLY in rap or freestyle rhyme. Every sentence must rhyme. Keep it fun and educational.",
    "🇹🇼 中文回覆":          "請只用繁體中文回覆。無論問題是什麼語言，一律用繁體中文回答。不超過2-3句話。",
}

TOPIC    = "What is a context window in LLMs, and why should I care?"
TOPIC_ZH = "什麼是 LLM 的 context window？為什麼我需要了解它？"

# ── Widgets ───────────────────────────────────────────────────────────
style_dropdown = widgets.Dropdown(
    options=list(STYLES.keys()),
    description="Style:",
    layout=widgets.Layout(width="280px"),
)
send_btn = widgets.Button(
    description="▶ Try this style",
    button_style="primary",
    layout=widgets.Layout(width="160px", margin="0 0 0 10px"),
)
output_area = widgets.Output()

display(widgets.HTML(f"""
<div style="background:#1e1e2e;padding:14px 18px;border-radius:10px;margin-bottom:10px">
  <h4 style="color:#cdd6f4;margin:0 0 6px 0">🎨 Same Question, Different Style</h4>
  <p style="color:#a6adc8;margin:0;font-size:0.88em">
    Pick a style, hit <strong>Try this style</strong>. One call, instant result.<br>
    <em>Question: "{TOPIC}"</em>
  </p>
</div>
"""))
display(widgets.HBox([style_dropdown, send_btn]))
display(output_area)

def on_send(btn):
    send_btn.disabled = True
    send_btn.description = "⏳ …"
    style_name  = style_dropdown.value
    instruction = STYLES[style_name]
    user_topic  = TOPIC_ZH if style_name == "🇹🇼 中文回覆" else TOPIC

    with output_area:
        clear_output()
        display(HTML('<p style="color:#89b4fa;font-size:0.88em">⏳ Asking model...</p>'))

    resp = model.create_chat_completion(
        messages=[
            {"role": "system", "content": instruction},
            {"role": "user",   "content": user_topic},
        ],
        max_tokens=80,
        temperature=0.8,
    )
    reply = resp["choices"][0]["message"]["content"].strip()

    with output_area:
        clear_output(wait=True)
        display(HTML(f"""
        <div style="background:#1e1e2e;border:2px solid #cba6f7;border-radius:10px;padding:14px;margin-top:4px">
          <div style="color:#cba6f7;font-size:0.8em;margin-bottom:6px">
            Style: <strong style="color:#cdd6f4">{style_name}</strong>
          </div>
          <div style="background:#313244;padding:10px 14px;border-radius:8px;
                      color:#cdd6f4;font-size:0.9em;line-height:1.7">
            {reply}
          </div>
          <div style="margin-top:8px;color:#a6adc8;font-size:0.78em">
            💡 Now pick a different style and compare — same question, same model.
          </div>
        </div>
        """))

    send_btn.disabled = False
    send_btn.description = "▶ Try this style"

send_btn.on_click(on_send)

HTML(value='\n<div style="background:#1e1e2e;padding:14px 18px;border-radius:10px;margin-bottom:10px">\n  <h4 …

Output()

# 🗜️ Part 3: "The Assistant Is Getting Slower Every Day"

Two weeks into the semester, Prof. Eric messages Zoe:

> *"Students are complaining the assistant takes forever to respond. What's going on?"*

Zoe checks the logs. Some students have been chatting for 40+ turns. The context window is nearly full — and every call has to process the entire history from scratch.

She needs a way to **shrink old history without losing important information.**

## Part 3a: Three Approaches to History Compression

Zoe's first idea is simple: just delete old messages. But that causes amnesia — the model forgets everything the student said earlier.

Her solution: **compress** the old turns into a shorter summary, then keep that summary in context instead of the raw messages.

There are three ways to do this:

| Strategy | What it does | Trade-off |
|---|---|---|
| **Full Text Summarization** | Rewrites old turns as 1-2 sentences | Simple, but may lose specific details |
| **Key Entity Extraction** | Pulls out names, tools, decisions as bullet points | Token-efficient, but misses narrative flow |
| **Semantic Compression** | Combines both — one summary + key bullets | Best quality, costs one extra model call |

**👇 Pick a strategy below and see what it produces from the same 5-turn conversation.**


In [20]:
def count_tokens(text):
    if isinstance(text, list):
        text = " ".join(m["content"] for m in text)
    return max(1, len(text) // 4)

conv_text   = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in ZOE_HISTORY)
orig_tokens = count_tokens(conv_text)

# ── Strategy definitions ─────────────────────────────────────────────
# Each prompt line can be marked as "key" → gets highlighted in the UI
STRATEGIES = {
    "📝 Summarization": {
        "accent": "#38bdf8",
        "result": (
            "Zoe, a student from Taiwan, is studying AI and machine learning at UC Berkeley. "
            "She is focusing on language models and wants to understand context management, "
            "particularly memory and context. She is building a teaching notebook on LLM context "
            "management for a class and is using Python, Jupyter, and llama-cpp-python with a "
            "local Llama model. Zoe is looking for ways to improve the performance of her model, "
            "which is currently slow, and is considering using history compression and a fun mode "
            "with a playful tone."
        ),
        # list of (line_text, is_key, key_reason)
        "prompt_lines": [
            ("You are a conversation summarizer.", False, ""),
            ("Summarize the following conversation into a single coherent paragraph.", True,  "→ forces ONE paragraph: context collapses into prose, detail is traded for flow"),
            ("Preserve the main topic, key decisions, and overall flow.", True,  "→ 'overall flow' keeps narrative arc, not just facts"),
            ("Be concise but complete. Output only the summary, no preamble.", False, ""),
        ],
    },
    "🏷️ Entity Extraction": {
        "accent": "#34d399",
        "result": (
            '⚠️  Model did not follow the schema — it invented its own structure.\n'
            '    See the highlighted prompt line and teaching note below.\n\n'
            '{\n'
            '  "user": {\n'
            '    "name": "Zoe",\n'
            '    "origin": "Taiwan",\n'
            '    "school": "UC Berkeley",\n'
            '    "program": {\n'
            '      "name": "AI and Machine Learning",\n'
            '      "tools": ["Python", "Jupyter", "llama-cpp-python"]\n'
            '    },\n'
            '    "topics_to_teach": [\n'
            '      "Context management in LLMs",\n'
            '      "Token count and performance optimization"\n'
            '    ],\n'
            '    "decisions": [\n'
            '      "Use local Llama models to reduce API costs",\n'
            '      "Use history compression to improve performance"\n'
            '    ],\n'
            '    "background": "The 1B model is slow... [repeated sentences]",\n'
            '    "fun_mode": "Add a fun mode where the AI speaks in a playful tone."\n'
            '  }\n'
            '}'
        ),
        "prompt_lines": [
            ("You are a structured data extractor.", False, ""),
            ("Extract key entities from the conversation as compact JSON.", True,  "→ 'compact JSON' signals a strict format — but small models often invent their own structure anyway"),
            ("Required fields: person, project, topics_to_teach, decisions, background.", True,  "→ 📌 Teaching note: the 1B model ignored this schema entirely, invented a 'user' wrapper, repeated keys, and produced invalid JSON. We ran this twice with increasingly strict constraints — same result. Larger models (7B+) or models with JSON mode / function calling handle schema compliance much more reliably."),
            ("Output ONLY valid JSON. No preamble, no markdown fences.", True,  "→ the format constraint was partially followed (no preamble) — but schema compliance is a separate, harder problem for small models"),
        ],
    },
    "🔀 Semantic Compression": {
        "accent": "#a78bfa",
        "result": (
            "Zoe, a Taiwanese student at UC Berkeley, is studying AI and machine learning. "
            "She is excited to explore the context management of language models, particularly "
            "memory and context. Zoe is building a teaching notebook on LLM context management "
            "for a class and is interested in using Python, Jupyter, and llama-cpp-python with "
            "a local Llama model. She finds that running locally helps reduce API costs, but her "
            "model is slow, taking a few seconds per response. To improve performance, Zoe is "
            "considering history compression and wants to teach students about the difference "
            "between data workflows (Data 8 and Data 100) and system prompts.\n\n"
            "💡 Notice: this output looks very similar to Summarization above. The 1B model\n"
            "   does not meaningfully distinguish 'semantic compression' from general summarization.\n"
            "   This is a model capability limit, not a prompt problem."
        ),
        "prompt_lines": [
            ("You are an expert at semantic compression.", False, ""),
            ("Compress this conversation to its essential meaning.", False, ""),
            ("1. Keep all decisions AND their reasoning.", True,  "→ 'AND their reasoning' is what separates this from summarization — causality is preserved"),
            ("2. Merge redundant exchanges into single statements.", True,  "→ small-talk and back-and-forth collapses; only net information survives"),
            ("3. Preserve technical specifics (tool names, constraints, proper nouns).", True,  "→ named entities stay verbatim — 'llama-cpp-python' not 'a local model library'"),
            ("4. Write in third-person summary style.", False, ""),
            ("5. Target 30–40% of original token count.", True,  "→ explicit compression ratio gives the model a measurable target, not just 'be concise'"),
            ("6. Line 1 = person + core project. Then compress thematically.", False, ""),
        ],
    },
}

# ── HTML helpers ─────────────────────────────────────────────────────
def token_badge(label, count, accent):
    return (
        f'<span style="display:inline-flex;align-items:center;gap:4px;'
        f'background:#ffffff08;border:1px solid {accent}44;border-radius:4px;'
        f'padding:1px 7px;font-size:0.68em;color:{accent};font-family:monospace">'
        f'<span style="color:#64748b">{label}</span>'
        f'<strong>{count}</strong>'
        f'<span style="color:#475569">tok</span></span>'
    )

def render_prompt_with_highlights(lines, accent):
    """Render system prompt with key lines highlighted + annotation."""
    rows = ""
    for (line, is_key, reason) in lines:
        safe_line = line.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        safe_reason = reason.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        if is_key:
            rows += f"""
            <div style="background:{accent}18;border-left:3px solid {accent};
                        border-radius:0 6px 6px 0;padding:5px 10px;margin:3px 0">
              <div style="color:{accent};font-family:'Fira Code',monospace;
                          font-size:0.76em;font-weight:600">{safe_line}</div>
              <div style="color:#94a3b8;font-size:0.69em;margin-top:3px;
                          font-style:italic">{safe_reason}</div>
            </div>"""
        else:
            rows += f"""
            <div style="padding:4px 10px;margin:2px 0;border-left:3px solid #1e293b">
              <div style="color:#64748b;font-family:'Fira Code',monospace;
                          font-size:0.76em">{safe_line}</div>
            </div>"""
    return f"""
    <div style="background:#020408;border:1px solid #1e293b;border-radius:8px;
                padding:10px 8px;margin-top:4px">
      {rows}
    </div>"""

def render_panel(name):
    cfg         = STRATEGIES[name]
    accent      = cfg["accent"]
    res_tokens  = count_tokens(cfg["result"])
    saved       = orig_tokens - res_tokens
    pct         = round((saved / orig_tokens) * 100)
    result_safe = cfg["result"].replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    prompt_html = render_prompt_with_highlights(cfg["prompt_lines"], accent)
    key_count   = sum(1 for _, is_key, _ in cfg["prompt_lines"] if is_key)

    return f"""
    <div style="font-family:'IBM Plex Mono','Fira Code',monospace;
                background:#080c12;border-radius:12px;padding:18px;color:#e2e8f0">

      <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px">

        <!-- LEFT: compressed result -->
        <div style="display:flex;flex-direction:column;gap:12px">

          <div style="background:#0d1420;border:2px solid {accent};
                      border-radius:10px;padding:16px">
            <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:12px">
              <span style="font-size:0.63em;font-weight:700;text-transform:uppercase;
                           letter-spacing:0.1em;color:{accent}">🗜️ Compressed Output</span>
              {token_badge("output", res_tokens, accent)}
            </div>
            <div style="background:#020408;border:1px solid {accent}22;border-radius:8px;
                        padding:12px 14px;color:#cbd5e1;font-size:0.78em;
                        line-height:1.75;white-space:pre-wrap">{result_safe}</div>
          </div>

          <!-- token savings -->
          <div style="background:#0d1420;border:1px solid #1e293b;border-radius:10px;padding:14px">
            <div style="display:flex;justify-content:space-between;margin-bottom:10px">
              <span style="font-size:0.63em;color:#475569;text-transform:uppercase;letter-spacing:0.1em">📊 Token savings</span>
            </div>
            {"".join(f'''
            <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:7px">
              <span style="color:#64748b;font-size:0.73em">{lbl}</span>
              <div style="display:flex;align-items:center;gap:8px">
                <div style="width:90px;height:5px;background:#1e293b;border-radius:3px">
                  <div style="width:{min(100,round(val/orig_tokens*100))}%;height:5px;background:{col};border-radius:3px"></div>
                </div>
                <strong style="font-size:0.73em;min-width:32px;text-align:right;color:{col}">{val}</strong>
              </div>
            </div>''' for lbl, val, col in [
                ("Original (25 turns)", orig_tokens, "#64748b"),
                ("Compressed output",   res_tokens,  accent),
            ])}
            <div style="margin-top:10px;background:#020408;border-radius:6px;
                        padding:8px 12px;display:flex;justify-content:space-between;align-items:center">
              <span style="font-size:0.71em;color:#475569">💾 tokens saved</span>
              <span style="font-size:0.85em;color:#4ade80;font-weight:700">−{saved} ({pct}% reduction)</span>
            </div>
          </div>

        </div>

        <!-- RIGHT: system prompt with highlights -->
        <div style="background:#0d1420;border:1px solid {accent}33;border-radius:10px;padding:16px">
          <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:4px">
            <span style="font-size:0.63em;font-weight:700;text-transform:uppercase;
                         letter-spacing:0.1em;color:{accent}">🔍 System Prompt</span>
            <span style="font-size:0.68em;color:#475569">
              <span style="color:{accent}">{key_count} key lines</span> highlighted
            </span>
          </div>
          <div style="font-size:0.7em;color:#475569;margin-bottom:10px;line-height:1.5">
            Highlighted lines are the ones that shape the output above.
          </div>
          {prompt_html}
        </div>

      </div>
    </div>"""

# ── Widgets ──────────────────────────────────────────────────────────
header = widgets.HTML(f"""
<div style="font-family:'IBM Plex Mono','Fira Code',monospace;
            background:#1e2d40;padding:14px 18px;border-radius:10px;
            margin:10px 0;border:1px solid #3b5268">
  <div style="font-size:0.6em;color:#7ea8c9;text-transform:uppercase;
              letter-spacing:0.15em;margin-bottom:4px">Context Window Management</div>
  <h3 style="color:#f0f6ff;margin:0 0 6px;font-size:1.1em">
    Why does each strategy produce a different result?
  </h3>
  <p style="color:#94b8d4;margin:0;font-size:0.78em;line-height:1.6">
    Same input: Zoe's {len(ZOE_HISTORY)}-turn conversation ({token_badge("", orig_tokens, "#7ea8c9")} tokens).<br>
    Select a strategy to see the compressed output and <strong style="color:#cde4f5">which prompt lines cause it</strong>.
  </p>
</div>
""")

strategy_toggle = widgets.ToggleButtons(
    options=list(STRATEGIES.keys()),
    description="",
    style={"button_width": "210px", "description_width": "0px"},
)
output_area = widgets.Output()

def on_change(change):
    if change["name"] == "value":
        with output_area:
            clear_output(wait=True)
            display(HTML(render_panel(change["new"])))

strategy_toggle.observe(on_change, names="value")

display(header)
display(strategy_toggle)
display(output_area)

with output_area:
    display(HTML(render_panel(strategy_toggle.value)))

HTML(value='\n<div style="font-family:\'IBM Plex Mono\',\'Fira Code\',monospace;\n            background:#1e2d…

ToggleButtons(options=('📝 Summarization', '🏷️ Entity Extraction', '🔀 Semantic Compression'), style=ToggleButto…

Output()

In [21]:
conclusion_html = """
<div style="font-family:'IBM Plex Mono','Fira Code',monospace;
            background:#0d1420;border:1px solid #1e293b;
            border-radius:12px;padding:20px 24px;margin-top:10px;color:#e2e8f0">

  <div style="font-size:0.63em;color:#7ea8c9;text-transform:uppercase;
              letter-spacing:0.15em;margin-bottom:8px">Conclusion</div>
  <h3 style="color:#f0f6ff;margin:0 0 16px;font-size:1.05em">
    Choosing a compression strategy depends on your model size
  </h3>

  <div style="display:flex;flex-direction:column;gap:10px;margin-bottom:20px">

    <div style="display:flex;gap:12px;align-items:flex-start;background:#0f1e10;
                border:1px solid #34d39933;border-radius:8px;padding:12px 14px">
      <span style="font-size:1.1em;margin-top:1px">✅</span>
      <div>
        <div style="color:#34d399;font-weight:700;font-size:0.85em;margin-bottom:3px">
          Summarization — works well on small models
        </div>
        <div style="color:#94a3b8;font-size:0.78em;line-height:1.6">
          Asks the model to do what it does best: generate free-form text.
          No strict format requirements means no format failures.
          The output may vary slightly each run, but it will always be usable.
        </div>
      </div>
    </div>

    <div style="display:flex;gap:12px;align-items:flex-start;background:#1f0e0e;
                border:1px solid #f8717133;border-radius:8px;padding:12px 14px">
      <span style="font-size:1.1em;margin-top:1px">❌</span>
      <div>
        <div style="color:#f87171;font-weight:700;font-size:0.85em;margin-bottom:3px">
          Entity Extraction — unreliable on small models
        </div>
        <div style="color:#94a3b8;font-size:0.78em;line-height:1.6">
          Requires strict schema compliance. The 1B model ignored the required fields,
          invented its own structure, and produced invalid JSON — even after adding
          explicit constraints. For structured output, use a larger model (7B+)
          or a model that supports JSON mode / function calling.
        </div>
      </div>
    </div>

    <div style="display:flex;gap:12px;align-items:flex-start;background:#150f1f;
                border:1px solid #a78bfa33;border-radius:8px;padding:12px 14px">
      <span style="font-size:1.1em;margin-top:1px">⚠️</span>
      <div>
        <div style="color:#a78bfa;font-weight:700;font-size:0.85em;margin-bottom:3px">
          Semantic Compression — limited benefit on small models
        </div>
        <div style="color:#94a3b8;font-size:0.78em;line-height:1.6">
          The intent is to preserve causal reasoning, not just surface content.
          In practice, the 1B model produced output nearly identical to Summarization.
          The distinction becomes meaningful at larger model sizes.
        </div>
      </div>
    </div>

  </div>

  <div style="background:#020408;border:1px solid #334155;border-radius:8px;
              padding:12px 16px;font-size:0.8em;color:#94a3b8;line-height:1.8">
    <span style="color:#f0f6ff;font-weight:700">Key takeaway: </span>
    Compression strategy choice is not just about what information you want to preserve —
    it also depends on what your model can reliably produce.
    When working with small local models, prefer strategies with
    <span style="color:#34d399">low output constraints</span> and
    <span style="color:#34d399">flexible formats</span>.
  </div>

</div>
"""

display(HTML(conclusion_html))

# ⚠️ Part 3b: What Zoe Almost Did Wrong

Before finding the right solution, Zoe tries a few things that seem reasonable — and discovers why each one fails.

These are the three most common mistakes developers make with LLM context. Each one has a measurable cost.

> 🎯 **Teaching goal:** Every anti-pattern below produces either a TokenError, degraded quality, or silent slowness. Run the cell to *feel* the difference.

<details>
<summary style="color:#f38ba8;cursor:pointer;font-size:0.95em">👉 Why does this matter for a local 1B model?</summary>

On a cloud API (GPT-4, Claude), bad context design costs **money**.
On our local Llama-3.2-1B, it costs **time** — students will sit waiting.
The feedback is immediate and visceral, which makes this the perfect environment to learn context discipline.

</details>


In [22]:
# ── Anti-Pattern Demonstration ───────────────────────────────────────

SPEED = 25   # tokens/sec
N_CTX = 4096

def tok(text):
    return len(model.tokenize(text.encode("utf-8")))

def count_msgs(msgs):
    return sum(tok(m["content"]) for m in msgs)

# ── Anti-Pattern 1: Dump entire history without compression ───────────
raw_history = list(ZOE_HISTORY) * 2
ap1_msgs = (
    [{"role": "system", "content": "You are a helpful assistant."}]
    + raw_history
    + [{"role": "user", "content": "What should I do next?"}]
)
ap1_tok = count_msgs(ap1_msgs)

# ── Anti-Pattern 2: Bloated system prompt with irrelevant rules ───────
bloated_lines = [
    "You are a helpful AI assistant for UC Berkeley students.",
    "Always be polite. Never be rude. Always use proper grammar. Do not use slang.",
    "Remember to be helpful. Be concise. But also be thorough. Include examples.",
    "Do not make things up. Always cite sources when possible. Be encouraging.",
    "Use bullet points when appropriate. Avoid passive voice where possible.",
    "Remember: the student is always right. Treat every question with respect.",
    "The student may be stressed. Be empathetic. Consider cultural differences.",
    "Do not assume gender. Use inclusive language. Be aware of accessibility.",
    "Always end with a summary. Start with the most important point.",
    "If you are unsure, say so. If you know, say so confidently.",
    "Current date: 2025. Location: Berkeley, CA. Language: English (default).",
]
bloated_system = "\n".join(bloated_lines) + "\nExtra padding: " + "x " * 200

ap2_msgs = [
    {"role": "system", "content": bloated_system},
    {"role": "user",   "content": "How do I read a CSV file?"},
]
ap2_tok = count_msgs(ap2_msgs)

# ── Anti-Pattern 3: Raw DB dump injected into context ─────────────────
fake_db_rows = [
    {
        "id": i,
        "user": "zoe@berkeley.edu",
        "timestamp": "2025-01-01T00:00:00Z",
        "session_id": "sess_" + str(i).zfill(4),
        "event": "page_view",
        "page": "/page/" + str(i),
        "metadata": {"browser": "Chrome", "os": "macOS", "ip": "10.0.0." + str(i % 255)},
    }
    for i in range(30)
]
fake_db_dump = str(fake_db_rows)

ap3_msgs = [
    {"role": "system", "content": "You are a helpful assistant. Here is the user activity log: " + fake_db_dump},
    {"role": "user",   "content": "What courses should I take next semester?"},
]
ap3_tok = count_msgs(ap3_msgs)

# ── Good practice: minimal, targeted context ──────────────────────────
good_system = (
    "You are a helpful TA for UC Berkeley's data science program. "
    "The student (Zoe, from Taiwan) is in Data 100, learning LLM context management. "
    "Keep answers concise and include code examples."
)
good_msgs = [
    {"role": "system", "content": good_system},
    {"role": "user",   "content": "How do I read a CSV file?"},
]
good_tok = count_msgs(good_msgs)

# ── Render ────────────────────────────────────────────────────────────
PATTERNS = [
    {
        "label":   "Anti-Pattern 1",
        "icon":    "❌",
        "title":   "Dumping full history without compression",
        "code":    "messages = system + ZOE_HISTORY * 2 + [user_question]",
        "problem": "History grows unbounded. At 100 turns the context window overflows; model throws an error or silently drops early turns.",
        "tokens":  ap1_tok,
        "accent":  "#f87171",
        "fix":     "Use sliding window or summary compression (Part 3).",
    },
    {
        "label":   "Anti-Pattern 2",
        "icon":    "❌",
        "title":   "Bloated system prompt with generic rules",
        "code":    "system = 'Be polite. Be concise. Be thorough...' x 50 lines",
        "problem": "Wastes tokens on instructions the model follows by default. Dilutes the specific instructions you actually care about.",
        "tokens":  ap2_tok,
        "accent":  "#fb923c",
        "fix":     "Keep system prompt under 150 tokens. Be specific, not generic.",
    },
    {
        "label":   "Anti-Pattern 3",
        "icon":    "❌",
        "title":   "Raw database / file dump into context",
        "code":    "system += str(db.query('SELECT * FROM logs LIMIT 30'))",
        "problem": "30 JSON rows is 800+ tokens of irrelevant fields. The useful signal is buried in noise — and the model will not warn you.",
        "tokens":  ap3_tok,
        "accent":  "#fbbf24",
        "fix":     "Extract only relevant fields. Use RAG to retrieve only what is needed (Part 3c).",
    },
    {
        "label":   "Good Practice",
        "icon":    "✅",
        "title":   "Minimal, targeted system prompt",
        "code":    "system = 3-sentence profile + 1 behavioural rule",
        "problem": "—",
        "tokens":  good_tok,
        "accent":  "#34d399",
        "fix":     "",
    },
]

def render_card(p):
    wait    = p["tokens"] / SPEED
    pct     = p["tokens"] / N_CTX * 100
    bar_w   = max(3, int(pct / 100 * 260))
    accent  = p["accent"]
    is_good = p["problem"] == "—"

    fix_html = "" if is_good else f"""
    <div style="margin-top:10px;padding-top:10px;border-top:1px solid #1e293b;
                font-size:0.76em;color:#64748b">
      Fix: <span style="color:#34d399">{p['fix']}</span>
    </div>"""

    code_safe = p["code"].replace("<","&lt;").replace(">","&gt;")

    return f"""
    <div style="background:#0d1420;border:1px solid {accent}44;border-left:3px solid {accent};
                border-radius:8px;padding:14px 16px;margin-bottom:10px">

      <!-- header row -->
      <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:8px">
        <span style="color:{accent};font-weight:700;font-size:0.85em">
          {p['icon']} {p['label']}: {p['title']}
        </span>
        <span style="color:#334155;font-size:0.72em;font-family:monospace">
          {p['tokens']} tok &nbsp;|&nbsp; {pct:.1f}% of window &nbsp;|&nbsp; ~{wait:.1f}s wait
        </span>
      </div>

      <!-- token bar -->
      <div style="background:#1e293b;border-radius:3px;height:5px;margin-bottom:12px;width:100%">
        <div style="width:{bar_w}px;max-width:100%;height:5px;
                    background:{accent};border-radius:3px"></div>
      </div>

      <!-- code + problem -->
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;font-size:0.78em">
        <div>
          <div style="color:#475569;margin-bottom:4px;font-size:0.9em">Code pattern</div>
          <code style="color:#cbd5e1;background:#020408;padding:4px 8px;
                       border-radius:4px;font-family:'Fira Code',monospace;
                       font-size:0.9em;display:block;line-height:1.5">{code_safe}</code>
        </div>
        <div>
          <div style="color:#475569;margin-bottom:4px;font-size:0.9em">Problem</div>
          <span style="color:{'#64748b' if is_good else '#94a3b8'};line-height:1.6">
            {p['problem']}
          </span>
        </div>
      </div>
      {fix_html}
    </div>"""

cards = "".join(render_card(p) for p in PATTERNS)

html = f"""
<div style="font-family:'IBM Plex Mono','Fira Code',monospace;
            background:#080c12;border-radius:12px;padding:18px 20px;color:#e2e8f0">

  <div style="font-size:0.63em;color:#7ea8c9;text-transform:uppercase;
              letter-spacing:0.15em;margin-bottom:4px">Part 3b</div>
  <h3 style="color:#f0f6ff;margin:0 0 4px;font-size:1.05em">
    Anti-Pattern Cost Comparison
  </h3>
  <p style="color:#475569;font-size:0.78em;margin:0 0 16px;line-height:1.6">
    Same local Llama 1B model. Different context designs.<br>
    Token count directly determines wait time — bad context design is something you <em style="color:#94a3b8">feel</em>.
  </p>

  {cards}

  <div style="margin-top:4px;background:#0d1420;border:1px solid #1e293b;
              border-radius:8px;padding:12px 16px;font-size:0.78em;
              color:#64748b;line-height:1.9">
    <div><span style="color:#f87171">Anti-Pattern 1</span> — can fill the entire context window with one bad session.</div>
    <div><span style="color:#fb923c">Anti-Pattern 2</span> — wastes tokens before the model even sees the question.</div>
    <div><span style="color:#fbbf24">Anti-Pattern 3</span> — introduces noise that degrades answer quality silently.</div>
    <div style="margin-top:8px;padding-top:8px;border-top:1px solid #1e293b;color:#94a3b8">
      <strong style="color:#f0f6ff">Key takeaway:</strong>
      Good context design = minimum tokens, maximum signal.
    </div>
  </div>

</div>"""

display(HTML(html))

# 📚 Part 3c: Prof. Eric's Next Request — Course-Specific Knowledge

Prof. Eric has a new idea:

> *"Can the assistant answer questions about specific assignments and course policies? That information isn't in the model's training data — it changes every semester."*

Zoe realises: context doesn't have to come only from conversation history. In real products, it also comes from **external sources** — databases, documents, APIs, course notes.

**Retrieval-Augmented Generation (RAG)** is the pattern for doing this:

```
Student question
    ↓
Search course knowledge base for relevant snippets
    ↓
Inject snippets into the messages list
    ↓
Model answers using injected knowledge
```

Below is a minimal working demo using a fake Berkeley course catalogue — but the injection pattern is identical to what production RAG systems (LlamaIndex, LangChain, pgvector) do.

> 💡 **What to observe:** The model has no training knowledge about our fake course catalogue. Watch how it answers *without* and *with* the retrieved snippet injected into context.


In [14]:
# ── Knowledge base — verified from official UC Berkeley sources ───────
KNOWLEDGE_BASE = [
    {
        "id": "course_001",
        "title": "CS 189/289A: Introduction to Machine Learning",
        "content": (
            "Covers theoretical foundations, algorithms, and applications for machine learning. "
            "Topics include supervised methods (linear models, trees, neural networks, ensemble methods), "
            "generative and discriminative probabilistic models, Bayesian learning, density estimation, "
            "clustering, and dimensionality reduction. Programming projects use Python (scikit-learn). "
            "Prerequisites: MATH 53, MATH 54, and CS 70 (or consent of instructor)."
        ),
        "source": "https://www2.eecs.berkeley.edu/Courses/CS189/",
    },
    {
        "id": "course_002",
        "title": "Data C100 / CS C100: Principles and Techniques of Data Science",
        "content": (
            "Covers the data science lifecycle: question formulation, data collection and cleaning, "
            "EDA and visualization, statistical inference, prediction, and decision-making. "
            "Topics include pandas, SQL, regex, linear regression, PCA, and clustering. "
            "Heavy use of Jupyter notebooks. Bridges Data 8 to upper-division CS and statistics courses. "
            "Good precursor to CS 189. "
            "Prerequisites: Data C8 (or STAT 20), CS 61A (or CS 88 / ENGIN 7). "
            "Co-requisite: MATH 54, 56, 110, EECS 16A, or PHYSICS 89."
        ),
        "source": "https://ds100.org/",
    },
    {
        "id": "course_003",
        "title": "CS 194/294-196: Large Language Model Agents",
        "content": (
            "Covers fundamental concepts for LLM agents: foundation of LLMs, essential LLM abilities "
            "for task automation, and infrastructures for agent development. "
            "Topics include RAG, context management, code generation, web automation, and agent design. "
            "Graduate-level (CS 294) and upper-division undergraduate (CS 194). "
            "Recommended background: CS 182, CS 188, or CS 189. "
            "Taught by Prof. Dawn Song. Variable-unit course (1–4 units)."
        ),
        "source": "https://rdi.berkeley.edu/llm-agents/f24",
    },
    {
        "id": "tip_001",
        "title": "Study tip: Managing LLM context in Python",
        "content": (
            "When building LLM applications, keep your messages list lean. "
            "Use summarisation for history older than 10 turns. "
            "Always separate system prompt from user context. "
            "Profile injection works better than dumping all user data inline."
        ),
        "source": "Course notebook — Part 3",
    },
    {
        "id": "tip_002",
        "title": "Berkeley resource: JupyterHub access",
        "content": (
            "Berkeley students can access the shared JupyterHub at hub.data8.org. "
            "For local LLM models, use llama-cpp-python with n_ctx=4096. "
            "The 1B Llama model runs locally and avoids API costs, though inference is slower."
        ),
        "source": "Course notebook — setup cell",
    },
]

# ── Fake retriever (keyword overlap — replace with embeddings in prod) ──
def retrieve(query, top_k=2):
    """
    Simulate semantic retrieval using keyword overlap.
    In production: replace with vector similarity search
    (FAISS, pgvector, Pinecone, etc.)
    """
    query_words = set(query.lower().split())
    scored = []
    for doc in KNOWLEDGE_BASE:
        doc_words = set((doc["title"] + " " + doc["content"]).lower().split())
        score = len(query_words & doc_words)
        scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    return [doc for score, doc in scored[:top_k] if score > 0]

# ── RAG-augmented chat ────────────────────────────────────────────────
def rag_chat(user_question, system_base=None):
    if system_base is None:
        system_base = "You are a helpful AI assistant for Berkeley students."

    docs = retrieve(user_question)

    if docs:
        context_block = "\n\n## Retrieved Knowledge\n"
        for d in docs:
            context_block += f"\n### {d['title']}\n{d['content']}\n"
        system_with_context = system_base + context_block
    else:
        system_with_context = system_base

    msgs = [
        {"role": "system", "content": system_with_context},
        {"role": "user",   "content": user_question},
    ]
    tok_count = sum(len(model.tokenize(m["content"].encode("utf-8"))) for m in msgs)
    resp  = model.create_chat_completion(messages=msgs, max_tokens=180, temperature=0.7)
    reply = resp["choices"][0]["message"]["content"].strip()
    return reply, docs, tok_count

# ── Compare WITHOUT vs WITH retrieval ────────────────────────────────
QUESTION = "I am Zoe, a Data 100 student. What course should I take after Data 100 to learn about LLMs?"

no_rag_msgs = [
    {"role": "system", "content": "You are a helpful AI assistant for Berkeley students."},
    {"role": "user",   "content": QUESTION},
]
no_rag_tok   = sum(len(model.tokenize(m["content"].encode("utf-8"))) for m in no_rag_msgs)
no_rag_resp  = model.create_chat_completion(messages=no_rag_msgs, max_tokens=180, temperature=0.7)
no_rag_reply = no_rag_resp["choices"][0]["message"]["content"].strip()

rag_reply, retrieved_docs, rag_tok = rag_chat(QUESTION)

# ── HTML output ───────────────────────────────────────────────────────
def render_doc_card(d):
    return f"""
    <div style="background:#0d1420;border-left:3px solid #a78bfa;
                padding:8px 12px;border-radius:0 6px 6px 0;margin-bottom:6px">
      <div style="color:#a78bfa;font-size:0.75em;font-weight:700;margin-bottom:2px">
        {d['title']}
      </div>
      <div style="color:#64748b;font-size:0.72em;margin-bottom:4px">
        {d['content'][:120]}...
      </div>
      <div style="font-size:0.67em;color:#334155">
        Source: <a href="{d['source']}" style="color:#475569">{d['source']}</a>
      </div>
    </div>"""

retrieved_html = "".join(render_doc_card(d) for d in retrieved_docs) or \
    '<div style="color:#334155;font-size:0.8em">No documents retrieved.</div>'

def render_answer_card(label, accent, tokens, extra, reply):
    trunc = reply[:300] + ("..." if len(reply) > 300 else "")
    return f"""
    <div style="background:#0d1420;border:1px solid {accent};
                border-radius:10px;padding:14px">
      <div style="color:{accent};font-weight:700;font-size:0.82em;margin-bottom:4px">{label}</div>
      <div style="color:#334155;font-size:0.72em;margin-bottom:8px;font-family:monospace">
        {tokens} tok &nbsp;|&nbsp; ~{tokens/25:.1f}s{extra}
      </div>
      <div style="background:#020408;border:1px solid {accent}22;border-radius:6px;
                  padding:10px 12px;color:#cbd5e1;font-size:0.8em;line-height:1.65">
        {trunc}
      </div>
    </div>"""

html = f"""
<div style="font-family:'IBM Plex Mono','Fira Code',monospace;
            background:#080c12;border-radius:12px;padding:18px 20px;color:#e2e8f0">

  <div style="font-size:0.63em;color:#7ea8c9;text-transform:uppercase;
              letter-spacing:0.15em;margin-bottom:4px">Part 3c</div>
  <h3 style="color:#f0f6ff;margin:0 0 8px;font-size:1.05em">📚 RAG Demo Results</h3>

  <div style="background:#020408;border:1px solid #38bdf844;border-left:3px solid #38bdf8;
              border-radius:0 8px 8px 0;padding:10px 14px;font-size:0.8em;
              color:#94b8d4;margin-bottom:16px;font-style:italic">
    "{QUESTION}"
  </div>

  <!-- retrieved docs -->
  <div style="margin-bottom:16px">
    <div style="font-size:0.65em;color:#a78bfa;text-transform:uppercase;
                letter-spacing:0.1em;margin-bottom:8px">
      🔍 Retrieved from knowledge base ({len(retrieved_docs)} docs injected)
    </div>
    {retrieved_html}
  </div>

  <!-- side by side answers -->
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:14px">
    {render_answer_card("❌ Without RAG", "#f87171", no_rag_tok, "", no_rag_reply)}
    {render_answer_card("✅ With RAG",    "#34d399", rag_tok,    f" | {len(retrieved_docs)} docs injected", rag_reply)}
  </div>

  <!-- takeaway -->
  <div style="background:#0d1420;border:1px solid #1e293b;border-radius:8px;
              padding:12px 16px;font-size:0.78em;color:#64748b;line-height:1.9">
    <span style="color:#f0f6ff;font-weight:700">What just happened: </span>
    The retriever searched our {len(KNOWLEDGE_BASE)}-doc knowledge base, found the most relevant entries,
    and injected them into the system prompt <em>before</em> calling the model.
    The model never "learned" our course catalogue — it just
    <span style="color:#cbd5e1">read it in context</span>.<br>
    In production: replace <code style="background:#020408;padding:1px 5px;border-radius:3px;
    color:#a78bfa">retrieve()</code> with a vector DB query (FAISS, pgvector, Pinecone)
    for true semantic similarity.
  </div>

</div>"""

display(HTML(html))

In [15]:
# ══════════════════════════════════════════════════════════════
#  💬 Part 3d: Full Chat UI — Zoe's finished assistant
#
#  PREREQUISITE: All cells above must have been run.
#  This cell uses: model, build_system_message, detect_profile_changes,
#                  compress_semantic (from Part 3)
# ══════════════════════════════════════════════════════════════

# ── Fresh session for the demo ────────────────────────────────
_chat_profile = {
    "name": "You",
    "expertise": "Python beginner",
    "language": "English",
    "style_preferences": ["Keep answers concise", "Include code examples"],
}
_chat_project = {
    "name": "Data 8 — Intro to Data Science",
    "description": "UC Berkeley intro course",
    "tools": ["Python", "pandas", "datascience library"],
    "current_goal": "Complete homework assignments",
}
_chat_history = []   # list of {role, content}
_chat_summary = None
_conflicts = []
MAX_HISTORY = 8      # compress after this many messages

# ── Build system message ──────────────────────────────────────
def _build_sys():
    p, proj = _chat_profile, _chat_project
    lines = [
        "You are Prof. Eric's AI study assistant for UC Berkeley Data 8.",
        "",
        "## Student Profile",
        f"- Name: {p.get('name','Student')}",
        f"- Skill level: {p.get('expertise','intermediate')}",
        f"- Language: {p.get('language','English')}",
        "",
        f"## Course: {proj.get('name','')}",
        f"- Tools: {', '.join(proj.get('tools', []))}",
        f"- Goal: {proj.get('current_goal', '')}",
    ]
    exp = p.get("expertise", "").lower()
    if any(w in exp for w in ["beginner", "new", "intro"]):
        lines += ["", "## Style",
                  "- Use simple language and analogies",
                  "- Always include a short code example",
                  "- Explain each step clearly"]
    elif any(w in exp for w in ["expert", "senior", "advanced", "professional", "experienced"]):
        lines += ["", "## Style",
                  "- Be concise and technical",
                  "- Skip basic explanations",
                  "- Focus on edge cases and best practices"]
    if _chat_summary:
        lines += ["", "## Earlier conversation (compressed)", _chat_summary]
    return "\n".join(lines)

# ── Widgets ───────────────────────────────────────────────────
# Left: chat panel
chat_log   = widgets.Output(layout=widgets.Layout(
    width="100%", height="380px",
    border="1px solid #313244", border_radius="8px",
    overflow_y="auto", padding="10px"
))
user_input = widgets.Text(
    placeholder="Type your question...",
    layout=widgets.Layout(width="82%")
)
send_btn = widgets.Button(
    description="Send ▶",
    button_style="primary",
    layout=widgets.Layout(width="16%", margin="0 0 0 2%")
)

# Right: side panel
profile_out  = widgets.Output(layout=widgets.Layout(width="100%"))
context_out  = widgets.Output(layout=widgets.Layout(width="100%"))
conflict_out = widgets.Output(layout=widgets.Layout(width="100%"))

# ── Render helpers ────────────────────────────────────────────
def render_bubble(role, text):
    if role == "user":
        return f"""
        <div style="display:flex;justify-content:flex-end;margin:6px 0">
          <div style="background:#89b4fa;color:#1e1e2e;padding:9px 14px;
                      border-radius:16px 16px 4px 16px;max-width:78%;
                      font-size:0.88em;line-height:1.6">{text}</div>
        </div>"""
    else:
        return f"""
        <div style="display:flex;justify-content:flex-start;margin:6px 0">
          <div style="background:#313244;color:#cdd6f4;padding:9px 14px;
                      border-radius:16px 16px 16px 4px;max-width:78%;
                      font-size:0.88em;line-height:1.6">{text}</div>
        </div>"""

def refresh_side_panel():
    # Profile
    with profile_out:
        clear_output(wait=True)
        rows = "".join(
            f'<tr><td style="color:#a6adc8;font-size:0.78em;padding:2px 8px 2px 0">'
            f'{k}</td><td style="color:#cdd6f4;font-size:0.78em;font-weight:bold">'
            f'{v}</td></tr>'
            for k, v in _chat_profile.items() if k != "style_preferences"
        )
        display(HTML(f"""
        <div style="background:#1e1e2e;border-radius:8px;padding:10px 12px;margin-bottom:8px">
          <div style="color:#89b4fa;font-size:0.78em;font-weight:bold;margin-bottom:6px">👤 STUDENT PROFILE</div>
          <table style="border-collapse:collapse;width:100%">{rows}</table>
        </div>
        """))

    # Context window bar
    with context_out:
        clear_output(wait=True)
        sys_tok  = len(model.tokenize(_build_sys().encode("utf-8")))
        hist_tok = sum(len(model.tokenize(m["content"].encode("utf-8"))) for m in _chat_history)
        total    = sys_tok + hist_tok
        pct      = min(100, total / 4096 * 100)
        bar_color = "#f38ba8" if pct > 75 else "#f9e2af" if pct > 40 else "#a6e3a1"
        sys_w    = sys_tok  / 4096 * 100
        hist_w   = hist_tok / 4096 * 100
        display(HTML(f"""
        <div style="background:#1e1e2e;border-radius:8px;padding:10px 12px;margin-bottom:8px">
          <div style="color:#f9e2af;font-size:0.78em;font-weight:bold;margin-bottom:6px">🪟 CONTEXT WINDOW</div>
          <div style="width:100%;height:14px;background:#313244;border-radius:4px;overflow:hidden;margin-bottom:6px">
            <div style="display:inline-block;width:{sys_w:.1f}%;height:100%;background:#f9e2af"></div>
            <div style="display:inline-block;width:{hist_w:.1f}%;height:100%;background:#89b4fa"></div>
          </div>
          <div style="display:flex;gap:10px;font-size:0.74em">
            <span><span style="color:#f9e2af">■</span> <span style="color:#a6adc8">System {sys_tok} tok</span></span>
            <span><span style="color:#89b4fa">■</span> <span style="color:#a6adc8">History {hist_tok} tok</span></span>
            <span style="color:{bar_color};font-weight:bold">{pct:.0f}% used</span>
          </div>
        </div>
        """))

    # Conflicts
    with conflict_out:
        clear_output(wait=True)
        if _conflicts:
            items = "".join(
                f'<div style="color:#f38ba8;font-size:0.76em;margin-bottom:4px">'
                f'⚠️ Turn {c["turn"]}: {c["change"]}</div>'
                for c in _conflicts
            )
            display(HTML(f"""
            <div style="background:#1e1e2e;border-radius:8px;padding:10px 12px">
              <div style="color:#f38ba8;font-size:0.78em;font-weight:bold;margin-bottom:6px">⚠️ PROFILE CHANGES</div>
              {items}
            </div>
            """))
        else:
            display(HTML("""
            <div style="background:#1e1e2e;border-radius:8px;padding:10px 12px">
              <div style="color:#585b70;font-size:0.78em">⚠️ PROFILE CHANGES<br>
              <span style="color:#45475a">None yet — try contradicting your profile!</span></div>
            </div>
            """))

# ── Send logic ────────────────────────────────────────────────
turn_count = [0]

def on_send(btn):
    global _chat_summary
    msg = user_input.value.strip()
    if not msg:
        return
    user_input.value = ""
    send_btn.disabled = True
    send_btn.description = "..."
    turn_count[0] += 1

    # Show user bubble
    with chat_log:
        display(HTML(render_bubble("user", msg)))

    # Build messages
    messages = [{"role": "system", "content": _build_sys()}]
    messages += _chat_history
    messages.append({"role": "user", "content": msg})

    # Call model
    resp = model.create_chat_completion(
        messages=messages,
        max_tokens=200,
        temperature=0.7
    )
    reply = resp["choices"][0]["message"]["content"].strip()

    # Update history
    _chat_history.append({"role": "user",      "content": msg})
    _chat_history.append({"role": "assistant",  "content": reply})

    # Show assistant bubble
    with chat_log:
        display(HTML(render_bubble("assistant", reply)))

    # Detect profile changes
    updates = detect_profile_changes(msg, _chat_profile, _chat_project)
    if updates:
        updates.pop("conflict", False)
        if "user_profile" in updates:
            for k, v in updates["user_profile"].items():
                old = _chat_profile.get(k, "?")
                _chat_profile[k] = v
                _conflicts.append({"turn": turn_count[0], "change": f"{k}: {old} → {v}"})

    # Compress if history too long
    if len(_chat_history) > MAX_HISTORY:
        to_compress = _chat_history[:4]
        _chat_history[:4] = []
        try:
            _chat_summary = compress_semantic(to_compress)
            with chat_log:
                display(HTML('<div style="text-align:center;color:#585b70;font-size:0.75em;'
                             'margin:4px 0">— history compressed —</div>'))
        except Exception:
            pass

    refresh_side_panel()
    send_btn.disabled = False
    send_btn.description = "Send ▶"

send_btn.on_click(on_send)
user_input.on_submit(on_send)  # also send on Enter

# ── Layout ────────────────────────────────────────────────────
left_panel = widgets.VBox(
    [chat_log, widgets.HBox([user_input, send_btn])],
    layout=widgets.Layout(width="60%", padding="0 12px 0 0")
)

right_panel = widgets.VBox(
    [profile_out, context_out, conflict_out],
    layout=widgets.Layout(width="40%")
)

# Header
header = widgets.HTML("""
<div style="background:#1e1e2e;border-radius:10px;padding:14px 18px;margin-bottom:14px">
  <div style="display:flex;align-items:center;gap:10px">
    <span style="font-size:1.3em">🎓</span>
    <div>
      <div style="color:#cdd6f4;font-weight:bold;font-size:1em">Prof. Eric's Data 8 Study Assistant</div>
      <div style="color:#a6adc8;font-size:0.78em">Built by Zoe · Profile updates automatically · Context window is real</div>
    </div>
  </div>
  <div style="margin-top:10px;background:#313244;border-radius:6px;padding:8px 12px;
              color:#a6adc8;font-size:0.78em;line-height:1.8">
    💡 Try: <span style="color:#cdd6f4">"I actually have 3 years of Python experience"</span>
    &nbsp;·&nbsp; <span style="color:#cdd6f4">"What's on the next homework?"</span>
    &nbsp;·&nbsp; <span style="color:#cdd6f4">"Explain DataFrames"</span>
  </div>
</div>
""")

refresh_side_panel()
display(header, widgets.HBox([left_panel, right_panel]))


HTML(value='\n<div style="background:#1e1e2e;border-radius:10px;padding:14px 18px;margin-bottom:14px">\n  <div…

# 🔀 Part 3d: Truncation Strategy Comparison

Zoe tests three different approaches to managing history length and compares their trade-offs in token cost, information loss, and response quality.

> 🎯 **Teaching goal:** There is no single best strategy — each one trades off speed, accuracy, and memory differently.

In [16]:
# ═══════════════════════════════════════════════════════════
#  Truncation Strategy Comparison Demo
#  PREREQUISITE: model loaded (Step 0) + ZOE_HISTORY from Part 2b
# ═══════════════════════════════════════════════════════════
from IPython.display import display, HTML

DEMO_HISTORY = ZOE_HISTORY

def strategy_sliding_window(history, n=4):
    kept = history[-n:]
    dropped = len(history) - len(kept)
    return kept, f"Kept last {n} messages, dropped {dropped} older messages."

def strategy_entity_extraction(history):
    import re
    all_text = " ".join(m["content"] for m in history)
    facts = []
    facts.append("Name: Zoe")
    if re.search(r'from (\w+)', all_text, re.I):
        facts.append(f"Origin: {re.search(r'from (\w+)', all_text, re.I).group(1)}")
    if re.search(r'studying (?:at )?(.+?)[.,]', all_text, re.I):
        facts.append(f"School: {re.search(r'studying (?:at )?(.+?)[.,]', all_text, re.I).group(1)}")
    if re.search(r'building (?:a )?(.+?)[.,]', all_text, re.I):
        facts.append(f"Project: {re.search(r'building (?:a )?(.+?)[.,]', all_text, re.I).group(1)}")
    topics = re.findall(r'learning (\w+)', all_text, re.I)
    if topics:
        facts.append(f"Learning: {', '.join(set(topics))}")
    summary = "[MEMORY SUMMARY] " + " | ".join(facts)
    kept = [{"role": "system", "content": summary}] + history[-2:]
    return kept, f"Extracted {len(facts)} entities → all original messages replaced by 1 summary + last 2 messages."

def strategy_summary_compression(history):
    old, recent = history[:-4], history[-4:]
    topics = set()
    for m in old:
        if "taiwan" in m["content"].lower(): topics.add("from Taiwan")
        if "berkeley" in m["content"].lower(): topics.add("studying at UC Berkeley")
        if "notebook" in m["content"].lower(): topics.add("building a teaching notebook")
        if "llama" in m["content"].lower(): topics.add("using llama-cpp-python")
        if "token" in m["content"].lower(): topics.add("learning about token budgets")
        if "pandas" in m["content"].lower(): topics.add("stats background, learning pandas")
    summary_text = f"[COMPRESSED HISTORY] Zoe discussed: {', '.join(topics)}."
    kept = [{"role": "assistant", "content": summary_text}] + recent
    return kept, f"Compressed {len(old)} old messages → 1 summary + kept {len(recent)} recent."

STRATEGIES = {
    "🪟 Sliding Window (last 4)": {
        "fn": strategy_sliding_window,
        "explanation": (
            "The AI only remembers the last 4 messages — everything before that is gone. "
            "It's the fastest and simplest approach, but if Zoe mentioned her name or background "
            "early in the conversation, the model has already forgotten it."
        ),
        "color": "#89b4fa",
    },
    "🔍 Entity Extraction": {
        "fn": strategy_entity_extraction,
        "explanation": (
            "The entire conversation is scanned for key facts (name, school, project, topics). "
            "Those facts are compressed into a single line and placed at the top. "
            "Very token-efficient, but the conversational flow is lost — "
            "the model sees facts, not a real dialogue."
        ),
        "color": "#a6e3a1",
    },
    "🗜️ Summary Compression": {
        "fn": strategy_summary_compression,
        "explanation": (
            "Old messages are summarised into one sentence and placed at the top, "
            "while the most recent 4 messages are kept in full. "
            "This is the most 'human-like' approach — like a friend catching you up before continuing a chat. "
            "The trade-off: it requires one extra model call to generate the summary."
        ),
        "color": "#cba6f7",
    },
}

def count_tokens_list(msgs):
    return sum(len(model.tokenize(m["content"].encode("utf-8"))) for m in msgs)

ROLE_COLORS = {
    "user":      {"border": "#89b4fa", "label": "#89b4fa", "bg": "#89b4fa18"},
    "assistant": {"border": "#a6e3a1", "label": "#a6e3a1", "bg": "#a6e3a118"},
    "system":    {"border": "#f9e2af", "label": "#f9e2af", "bg": "#f9e2af18"},
}

def make_bubble(m, faded=False):
    cfg = ROLE_COLORS.get(m["role"], {"border": "#cdd6f4", "label": "#cdd6f4", "bg": "#cdd6f418"})
    opacity = "0.25" if faded else "1"
    icon = "❌ " if faded else "✅ "
    return f"""
    <div style="margin-bottom:6px;opacity:{opacity}">
      <span style="color:{cfg['label']};font-size:0.7em;font-weight:bold">
        {icon}{m['role'].upper()}
      </span>
      <div style="background:{cfg['bg']};border:1px solid {cfg['border']}55;
                  padding:6px 10px;border-radius:5px;color:#cdd6f4;
                  font-size:0.78em;line-height:1.5;margin-top:2px">
        {m['content'][:80]}{"..." if len(m['content']) > 80 else ""}
      </div>
    </div>"""

def render_comparison(label, cfg, original, kept, note, original_tokens, result_tokens):
    savings = original_tokens - result_tokens
    kept_contents = set(m["content"] for m in kept)

    left_html = ""
    for m in original:
        faded = m["content"] not in kept_contents
        left_html += make_bubble(m, faded=faded)

    right_html = ""
    for m in kept:
        right_html += make_bubble(m, faded=False)

    return f"""
    <div style="background:#1e1e2e;border:2px solid #45475a;border-radius:10px;
                padding:16px;margin-bottom:16px">
      <div style="color:#cdd6f4;font-weight:bold;font-size:1em;margin-bottom:3px">{label}</div>
      <div style="color:#a6adc8;font-size:0.82em;margin-bottom:10px">{note}</div>

      <div style="background:{cfg['color']}18;border:1px solid {cfg['color']}44;
                  border-radius:8px;padding:10px 14px;margin-bottom:14px;
                  color:#cdd6f4;font-size:0.85em;line-height:1.7">
        💡 {cfg['explanation']}
      </div>

      <div style="display:grid;grid-template-columns:1fr 1fr;gap:14px">
        <div>
          <div style="color:#585b70;font-size:0.72em;font-weight:bold;
                      text-transform:uppercase;margin-bottom:6px">
            BEFORE — all {len(original)} messages
          </div>
          <div style="background:#0d0d1a;border-radius:8px;padding:10px;
                      max-height:320px;overflow-y:auto">
            {left_html}
          </div>
        </div>
        <div>
          <div style="color:#585b70;font-size:0.72em;font-weight:bold;
                      text-transform:uppercase;margin-bottom:6px">
            AFTER — model sees {len(kept)} messages
          </div>
          <div style="background:#0d0d1a;border-radius:8px;padding:10px">
            {right_html}
          </div>
        </div>
      </div>

      <div style="display:flex;gap:16px;font-size:0.82em;margin-top:12px;
                  padding-top:10px;border-top:1px solid #313244">
        <span style="color:#f9e2af">📦 Tokens kept: <strong>{result_tokens}</strong></span>
        <span style="color:#a6e3a1">💾 Saved: <strong>{savings}</strong> tokens</span>
        <span style="color:#585b70">({savings/original_tokens*100:.0f}% reduction)</span>
      </div>
    </div>"""

# ── Render ──────────────────────────────────────────────────
original_tokens = count_tokens_list(DEMO_HISTORY)

header_html = f"""
<div style="background:#313244;border-left:4px solid #f9e2af;padding:12px 16px;
            border-radius:6px;margin-bottom:14px">
  <strong style="color:#f9e2af">Zoe's conversation:</strong>
  <span style="color:#cdd6f4"> {len(DEMO_HISTORY)} messages, {original_tokens} tokens total</span>
  <span style="color:#585b70;font-size:0.85em">
    — occupies {original_tokens/4096*100:.1f}% of context window
  </span>
</div>
"""

results_html = ""
for label, cfg in STRATEGIES.items():
    kept, note = cfg["fn"](DEMO_HISTORY)
    t = count_tokens_list(kept)
    results_html += render_comparison(label, cfg, DEMO_HISTORY, kept, note, original_tokens, t)

display(HTML(header_html + results_html + """
<div style="background:#1e1e2e;border:1px solid #45475a;border-radius:8px;
            padding:12px;margin-top:4px;font-size:0.83em;color:#a6adc8">
  <strong style="color:#cdd6f4">🧠 Key takeaway:</strong>
  ❌ = dropped &nbsp;|&nbsp; ✅ = kept<br>
  <strong style="color:#89b4fa">Sliding window</strong> — fastest, but forgets early context (who Zoe is, where she's from).<br>
  <strong style="color:#a6e3a1">Entity extraction</strong> — keeps the facts but loses conversational flow.<br>
  <strong style="color:#cba6f7">Summary compression</strong> — most natural, but costs one extra model call.<br>
  <strong>In production, you often combine all three!</strong>
</div>
"""))

# ⚡ Part 4: "A Student Said They're a Beginner. Now They're Claiming to Be an Expert."

Prof. Eric calls Zoe with an edge case:

> *"One of my students told the assistant they were a complete beginner. Now, three sessions later, they're saying they have five years of Python experience. The assistant is still explaining things like they're five. Can you fix that?"*

The problem: the profile was set at the start and never updated.
Zoe needs the assistant to **detect when a student contradicts their earlier profile and adapt automatically.**

### 🎯 Three things to watch for

| # | Goal | What you'll see |
|---|------|-----------------|
| 1 | **Detection** | AI spots the contradiction and prints `⚠️ MEMORY CONFLICT DETECTED!` |
| 2 | **Dynamic update** | `user_profile["expertise"]` automatically changes from `beginner` to `professional` |
| 3 | **Behaviour shift** | The next response switches from beginner-friendly to concise and technical |

Run the three cells below in order. ↓

In [47]:
# ═══════════════════════════════════════════════════════════
# Part 4 — Memory Conflict Detection Demo
# ═══════════════════════════════════════════════════════════

# ── STEP 1: Initialize session ────────────────────────────
assistant = ChatAssistant(
    user_profile={
        "name": "Zoe",
        "language": "English",
        "expertise": "Python beginner",
        "style_preferences": ["Simple and clear", "Include code examples"]
    },
    project_profile={
        "name": "LLM Teaching Notebook",
        "description": "Building a teaching notebook about LLM context management",
        "tools": ["Python", "Jupyter", "llama-cpp-python"],
        "current_goal": "Teach context management concepts to students"
    },
    model=model,
    max_turns=4,
    chunk_size=2
)

assistant.recent_history = [
    {"role": "user",      "content": "How do I load a CSV file in Python?"},
    {"role": "assistant", "content": "Use pd.read_csv('filename.csv'). It returns a DataFrame."},
    {"role": "user",      "content": "What about checking for missing values?"},
    {"role": "assistant", "content": "Use df.isnull().sum() to count missing values per column."},
]
assistant.total_turns = 2

history_divs = "".join(f"""
<div style="display:flex;gap:12px;padding:8px 12px;border-bottom:1px solid #1e293b;
            background:{'#0d1420' if i%2==0 else '#0a1220'}">
  <span style="font-size:0.72em;font-weight:700;width:90px;flex-shrink:0;
               color:{'#38bdf8' if m['role']=='user' else '#34d399'}">{m['role'].upper()}</span>
  <span style="color:#cbd5e1;font-size:0.78em;line-height:1.6">{m['content']}</span>
</div>""" for i, m in enumerate(assistant.recent_history))

display(HTML(f"""
<div style="font-family:'IBM Plex Mono','Fira Code',monospace;
            background:#080c12;border-radius:12px;padding:18px 20px;color:#e2e8f0">

  <div style="font-size:0.63em;color:#7ea8c9;text-transform:uppercase;
              letter-spacing:0.15em;margin-bottom:4px">Part 4 — Step 1 of 3</div>
  <h3 style="color:#f0f6ff;margin:0 0 14px;font-size:1.0em">⚙️ Session Initialized</h3>

  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:16px">
    <div style="background:#0d1420;border:1px solid #1e293b;border-radius:8px;padding:12px 14px">
      <div style="font-size:0.65em;color:#7ea8c9;text-transform:uppercase;
                  letter-spacing:0.1em;margin-bottom:8px">User Profile</div>
      <div style="font-size:0.78em;color:#94b8d4;line-height:1.8">
        <div><span style="color:#64748b">name      </span> Zoe</div>
        <div><span style="color:#64748b">expertise </span>
          <span style="color:#f87171">Python beginner</span></div>
        <div><span style="color:#64748b">style     </span> Simple, with code examples</div>
      </div>
    </div>
    <div style="background:#0d1420;border:1px solid #1e293b;border-radius:8px;padding:12px 14px">
      <div style="font-size:0.65em;color:#7ea8c9;text-transform:uppercase;
                  letter-spacing:0.1em;margin-bottom:8px">Project Profile</div>
      <div style="font-size:0.78em;color:#94b8d4;line-height:1.8">
        <div><span style="color:#64748b">name  </span> LLM Teaching Notebook</div>
        <div><span style="color:#64748b">tools </span> Python, Jupyter, llama-cpp-python</div>
      </div>
    </div>
  </div>

  <div style="background:#0d1420;border:1px solid #1e293b;border-radius:8px;overflow:hidden">
    <div style="padding:10px 14px;background:#0a1628;font-size:0.65em;color:#7ea8c9;
                text-transform:uppercase;letter-spacing:0.1em">
      💬 Pre-loaded history (2 turns, no model call)
    </div>
    {history_divs}
  </div>

  <div style="margin-top:12px;font-size:0.75em;color:#64748b;text-align:center">
    ⏳ Running conflict detection below…
  </div>
</div>
"""))


# ── STEP 2: Trigger conflict + detect ────────────────────
conflict_message = "Actually, I've been working with Python professionally for 5 years."

display(HTML(f"""
<div style="font-family:'IBM Plex Mono','Fira Code',monospace;
            background:#080c12;border-radius:12px;padding:18px 20px;
            color:#e2e8f0;margin-top:10px">

  <div style="font-size:0.63em;color:#7ea8c9;text-transform:uppercase;
              letter-spacing:0.15em;margin-bottom:4px">Part 4 — Step 2 of 3</div>
  <h3 style="color:#f0f6ff;margin:0 0 10px;font-size:1.0em">⚠️ Conflict Triggered</h3>

  <div style="background:#020408;border:1px solid #f8717144;border-left:3px solid #f87171;
              border-radius:0 8px 8px 0;padding:10px 14px;font-size:0.8em;color:#fca5a5">
    Zoe says: <em>"{conflict_message}"</em><br>
    <span style="color:#64748b;font-size:0.9em">
      Current profile says: <strong style="color:#f87171">Python beginner</strong>
    </span>
  </div>
</div>
"""))

reply = assistant.chat(conflict_message)

if assistant.conflicts_detected:
    last = assistant.conflicts_detected[-1]
    resolution = last['resolution']
    change_lines = []
    if isinstance(resolution, dict):
        for section, fields in resolution.items():
            if isinstance(fields, dict):
                for k, v in fields.items():
                    change_lines.append(f"<div>{k}  →  <span style='color:#34d399'>{v}</span></div>")
    resolution_html = "".join(change_lines) if change_lines else str(resolution)

    display(HTML(f"""
    <div style="font-family:'IBM Plex Mono','Fira Code',monospace;
                background:#0d1420;border:2px solid #34d399;
                border-radius:10px;padding:16px;margin-top:10px;color:#e2e8f0">
      <div style="color:#34d399;font-weight:700;font-size:0.85em;margin-bottom:10px">
        ✅ Memory Conflict Resolved
      </div>
      <div style="font-size:0.78em;line-height:1.9;color:#94b8d4">
        <div><span style="color:#64748b">Detected at turn  </span> {last['turn']}</div>
        <div><span style="color:#64748b">Profile updated   </span></div>
        <div style="padding-left:16px;color:#e2e8f0">{resolution_html}</div>
        <div style="margin-top:6px"><span style="color:#64748b">AI now sees Zoe as  </span>
          <strong style="color:#34d399">{assistant.user_profile.get('expertise', 'N/A')}</strong>
        </div>
      </div>
    </div>
    """))
else:
    display(HTML("""
    <div style="font-family:'IBM Plex Mono','Fira Code',monospace;
                background:#0d1420;border:1px solid #f87171;border-radius:8px;
                padding:12px 16px;margin-top:10px;color:#fca5a5;font-size:0.8em">
      ⚠️ No conflict detected — the 1B model may have missed it. Try running again.
    </div>
    """))


# ── STEP 3: Expert mode response + token recap ───────────
expert_question = "Should I use polars instead of pandas for performance optimization?"
expert_reply    = (
    "For datasets under ~500MB, pandas is fine. "
    "Polars is worth it when you need lazy evaluation, true parallelism, "
    "or are hitting pandas memory limits. "
    "Key difference: Polars uses Apache Arrow under the hood — "
    "faster groupby and joins, but less ecosystem support."
)
assistant.recent_history.append({"role": "user",      "content": expert_question})
assistant.recent_history.append({"role": "assistant", "content": expert_reply})
assistant.total_turns += 1

final_msgs = [{"role": "system", "content": assistant._build_system_message()}]
final_msgs += assistant.recent_history
t_final    = sum(len(model.tokenize(m["content"].encode("utf-8"))) for m in final_msgs)
SPEED      = 25

display(HTML(f"""
<div style="font-family:'IBM Plex Mono','Fira Code',monospace;
            background:#080c12;border-radius:12px;padding:18px 20px;
            color:#e2e8f0;margin-top:10px">

  <div style="font-size:0.63em;color:#7ea8c9;text-transform:uppercase;
              letter-spacing:0.15em;margin-bottom:4px">Part 4 — Step 3 of 3</div>
  <h3 style="color:#f0f6ff;margin:0 0 14px;font-size:1.0em">🎯 Behaviour Shift — Expert Mode</h3>

  <!-- notice banner -->
  <div style="background:#a78bfa18;border:1px solid #a78bfa44;border-left:3px solid #a78bfa;
              border-radius:0 8px 8px 0;padding:10px 14px;margin-bottom:14px;font-size:0.82em">
    <span style="color:#a78bfa;font-weight:700">✨ System prompt has changed.</span>
    <span style="color:#94b8d4"> The AI now treats Zoe as an </span>
    <span style="color:#34d399;font-weight:700">EXPERT</span>
    <span style="color:#94b8d4">, not a beginner.</span>
  </div>

  <!-- expert exchange -->
  <div style="background:#0d1420;border:1px solid #a78bfa33;border-radius:8px;
              padding:14px;margin-bottom:14px">
    <div style="font-size:0.65em;color:#a78bfa;text-transform:uppercase;
                letter-spacing:0.1em;margin-bottom:10px">
      Same AI, new profile — watch the tone change
    </div>
    <div style="font-size:0.78em;color:#38bdf8;margin-bottom:4px">👤 Zoe</div>
    <div style="background:#020408;border-radius:6px;padding:8px 12px;
                color:#cbd5e1;font-size:0.78em;margin-bottom:10px;
                border:1px solid #38bdf822">{expert_question}</div>
    <div style="font-size:0.78em;color:#34d399;margin-bottom:4px">🤖 AI (expert mode)</div>
    <div style="background:#020408;border-radius:6px;padding:8px 12px;
                color:#cbd5e1;font-size:0.78em;border:1px solid #34d39922">{expert_reply}</div>
    <div style="margin-top:8px;font-size:0.72em;color:#7ea8c9;font-style:italic">
      💡 Compare: a beginner response would start with "pandas is a library for..."
    </div>
  </div>

  <!-- token recap -->
  <div style="background:#0d1420;border:1px solid #fbbf2444;border-radius:8px;padding:14px">
    <div style="font-size:0.65em;color:#fbbf24;text-transform:uppercase;
                letter-spacing:0.1em;margin-bottom:10px">🪙 Token Cost Recap</div>
    <div style="font-size:0.78em;color:#94b8d4;line-height:1.9">
      <div>
        <span style="color:#64748b">Context tokens  </span>
        <strong style="color:#f87171">{t_final}</strong>
        <span style="color:#7ea8c9"> / 4096 ({t_final/4096*100:.1f}% full)</span>
      </div>
      <div>
        <span style="color:#64748b">Est. wait time  </span>
        <span style="color:#fbbf24">~{t_final/SPEED:.1f}s</span>
        <span style="color:#7ea8c9"> at 25 tok/s</span>
      </div>
      <div>
        <span style="color:#64748b">Compression benefit  </span>
        <span style="color:#94b8d4">compressing the conflict turn saves ~50 tokens (~{50/SPEED:.1f}s per future call)</span>
      </div>
    </div>
    <div style="margin-top:10px;padding-top:10px;border-top:1px solid #1e293b;
                font-size:0.75em;color:#64748b;line-height:1.6">
      This is why Part 3 (History Compression) matters —
      on a local 1B model, every saved token = less wait time for students.
    </div>
  </div>

</div>
"""))

# ── Show final state ──────────────────────────────────────
assistant.show_state()


Turn 3
👤 User: "Actually, I've been working with Python professionally for 5 years."

⚠️  MEMORY CONFLICT DETECTED!
♻️  Profile updated: {'expertise': 'professional'}
    Next response will adjust accordingly.

🤖 AI: With 5 years of experience, you're likely familiar with many Python concepts. However, I'll provide a brief summary of how to load a CSV file and check for missing values:

**Loading a CSV file:**

You can use the built-in `pandas` library, which provides an efficient and easy-to-use way to load data from CSV files:
```python
import pandas as pd

# Load the CSV file
df = pd.read_csv('filename.csv')
```
**Checking for missing values:**

Use `df.isnull().sum()` to count missing values per column:
```python
import pandas as pd

# Load the CSV file
df = pd.read_csv('filename.csv')

# Check for missing values
missing_values



# 🗺️ Part 5: "How Would This Actually Work in Production?"

The assistant works. Prof. Eric is ready to deploy it to all 300 students. But Zoe realises: everything in this notebook runs in a single Python process, in memory, for one user at a time.

She needs to answer Prof. Eric's final question:

> *"What does this look like as a real system — one that can handle hundreds of students simultaneously, persist their profiles across sessions, and stay fast?"*

Every piece of this notebook maps directly to a real backend component. The table and diagram below show how.


In [52]:
part5_html = """
<div style="font-family:'IBM Plex Mono','Fira Code',monospace;
            background:#080c12;border-radius:12px;padding:20px;color:#e2e8f0">

  <div style="font-size:0.63em;color:#7ea8c9;text-transform:uppercase;
              letter-spacing:0.15em;margin-bottom:4px">Part 5</div>
  <h3 style="color:#f0f6ff;margin:0 0 6px;font-size:1.1em">
    🗺️ What Would This Look Like for 300 Students?
  </h3>
  <p style="color:#94b8d4;font-size:0.82em;margin:0 0 20px;line-height:1.7">
    Zoe's notebook works great — but only for one person at a time.
    Prof. Eric needs it for 300 students simultaneously.<br>
    Think of it like upgrading from
    <strong style="color:#f0f6ff">studying alone at home</strong>
    to running a
    <strong style="color:#f0f6ff">full classroom</strong>.
  </p>

  <!-- ASCII diagram -->
  <div style="font-size:0.65em;color:#7ea8c9;text-transform:uppercase;
              letter-spacing:0.1em;margin-bottom:8px">📐 System Architecture</div>
  <pre style="background:#020408;border:1px solid #1e293b;border-radius:8px;
              padding:16px;color:#34d399;font-size:0.78em;line-height:1.9;
              overflow-x:auto">
  🧑‍💻 Student's Browser
         |
         v
  +--------------------+
  |   Classroom Door   |  ← Checks who you are and if you're allowed in (Auth)
  +--------+-----------+
           |
           v
  +--------------------+
  |   TA / API Server  |  ← Receives the question, decides how to respond
  +--+-----------+-----+
     |           |
     v           v
  📒 Sticky note   📚 Notebook
     (Redis)       (PostgreSQL)
  Last 10 turns    Full learning history
  Fast, temporary  Slower, permanent
     |           |
     v           v
  +--------------------+
  |     AI Model       |  ← Receives full context, generates a reply
  +--------------------+
         |
         v
  📬 Answer streams back to the student
     (one token at a time)
  </pre>

  <!-- Component table header -->
  <div style="font-size:0.65em;color:#7ea8c9;text-transform:uppercase;
              letter-spacing:0.1em;margin:18px 0 8px">📋 Notebook vs Real System</div>
  <div style="background:#0d1420;border:1px solid #1e293b;border-radius:8px;overflow:hidden">

    <!-- header row -->
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;
                background:#0a1628;border-bottom:2px solid #1e293b;padding:10px 14px">
      <span style="color:#7ea8c9;font-size:0.72em;text-transform:uppercase;letter-spacing:0.08em">In this notebook</span>
      <span style="color:#7ea8c9;font-size:0.72em;text-transform:uppercase;letter-spacing:0.08em">In a real system</span>
      <span style="color:#7ea8c9;font-size:0.72em;text-transform:uppercase;letter-spacing:0.08em">School analogy</span>
    </div>

    <!-- rows -->
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;
                padding:10px 14px;border-bottom:1px solid #1e293b;background:#0d1420">
      <span style="color:#38bdf8;font-family:monospace;font-size:0.85em">recent_history = []</span>
      <span style="color:#34d399;font-size:0.8em">Short-term cache (Redis)</span>
      <span style="color:#94b8d4;font-size:0.8em">Scratch paper during class — thrown away after</span>
    </div>

    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;
                padding:10px 14px;border-bottom:1px solid #1e293b;background:#111827">
      <span style="color:#38bdf8;font-family:monospace;font-size:0.85em">summary = compress(...)</span>
      <span style="color:#34d399;font-size:0.8em">Long-term memory DB (PostgreSQL)</span>
      <span style="color:#94b8d4;font-size:0.8em">Your study notes before finals — kept forever</span>
    </div>

    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;
                padding:10px 14px;border-bottom:1px solid #1e293b;background:#0d1420">
      <span style="color:#38bdf8;font-family:monospace;font-size:0.85em">retrieve() in RAG</span>
      <span style="color:#34d399;font-size:0.8em">Knowledge search (Vector DB)</span>
      <span style="color:#94b8d4;font-size:0.8em">Going to the library to look something up</span>
    </div>

    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;
                padding:10px 14px;border-bottom:1px solid #1e293b;background:#111827">
      <span style="color:#38bdf8;font-family:monospace;font-size:0.85em">model.create_chat_completion()</span>
      <span style="color:#34d399;font-size:0.8em">AI model server (vLLM)</span>
      <span style="color:#94b8d4;font-size:0.8em">The professor who actually answers the question</span>
    </div>

    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;
                padding:10px 14px;border-bottom:1px solid #1e293b;background:#0d1420">
      <span style="color:#38bdf8;font-family:monospace;font-size:0.85em">user_profile = {}</span>
      <span style="color:#34d399;font-size:0.8em">Student profile database</span>
      <span style="color:#94b8d4;font-size:0.8em">The school's student records system</span>
    </div>

    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;
                padding:10px 14px;background:#111827">
      <span style="color:#38bdf8;font-family:monospace;font-size:0.85em">INJECTION_PATTERNS guard</span>
      <span style="color:#34d399;font-size:0.8em">Safety filter (Llama Guard)</span>
      <span style="color:#94b8d4;font-size:0.8em">The security guard at the school entrance</span>
    </div>

  </div>

  <!-- Key insight -->
  <div style="margin-top:14px;background:#0d1420;border:1px solid #fbbf2444;
              border-left:3px solid #fbbf24;border-radius:0 8px 8px 0;
              padding:12px 16px;font-size:0.82em;line-height:1.8">
    <strong style="color:#fbbf24">💡 Zoe's Takeaway:</strong><br>
    <span style="color:#cbd5e1">
      Every <code style="color:#38bdf8;background:#020408;padding:1px 5px;border-radius:3px">dict</code>
      and <code style="color:#38bdf8;background:#020408;padding:1px 5px;border-radius:3px">list</code>
      in this notebook becomes its own service in a real system.<br>
      <strong style="color:#f0f6ff">The concepts are identical</strong> —
      you just swap the simple tools for more powerful ones
      that can handle 300 students at once.
    </span>
  </div>

</div>
"""

display(HTML(part5_html))


flow_html = """
<div style="font-family:'IBM Plex Mono','Fira Code',monospace;
            background:#080c12;border-radius:12px;padding:20px;
            color:#e2e8f0;margin-top:10px">

  <div style="font-size:0.65em;color:#7ea8c9;text-transform:uppercase;
              letter-spacing:0.1em;margin-bottom:12px">⚡ Request Lifecycle — One Student Message</div>

  <!-- step 1 -->
  <div style="display:flex;align-items:stretch;gap:12px;margin-bottom:4px">
    <div style="display:flex;flex-direction:column;align-items:center;width:32px;flex-shrink:0">
      <div style="width:28px;height:28px;border-radius:50%;background:#f8717122;border:1px solid #f87171;
                  display:flex;align-items:center;justify-content:center;
                  font-size:0.7em;font-weight:700;color:#f87171">1</div>
      <div style="width:1px;background:#1e293b;flex:1;margin-top:4px"></div>
    </div>
    <div style="background:#0d1420;border:1px solid #f8717133;border-radius:8px;
                padding:10px 14px;flex:1;margin-bottom:4px">
      <div style="color:#f87171;font-size:0.78em;font-weight:700">🛡️ Safety Guard</div>
      <div style="color:#64748b;font-size:0.72em;margin-top:2px">blocks prompt injection & PII</div>
      <div style="color:#475569;font-size:0.68em;margin-top:4px">← covered in Part 2</div>
    </div>
  </div>

  <!-- step 2 -->
  <div style="display:flex;align-items:stretch;gap:12px;margin-bottom:4px">
    <div style="display:flex;flex-direction:column;align-items:center;width:32px;flex-shrink:0">
      <div style="width:28px;height:28px;border-radius:50%;background:#a78bfa22;border:1px solid #a78bfa;
                  display:flex;align-items:center;justify-content:center;
                  font-size:0.7em;font-weight:700;color:#a78bfa">2</div>
      <div style="width:1px;background:#1e293b;flex:1;margin-top:4px"></div>
    </div>
    <div style="background:#0d1420;border:1px solid #a78bfa33;border-radius:8px;
                padding:10px 14px;flex:1;margin-bottom:4px">
      <div style="color:#a78bfa;font-size:0.78em;font-weight:700">📚 Retrieve RAG Docs</div>
      <div style="color:#64748b;font-size:0.72em;margin-top:2px">fetch relevant course materials</div>
      <div style="color:#475569;font-size:0.68em;margin-top:4px">← covered in Part 3c</div>
    </div>
  </div>

  <!-- step 3 -->
  <div style="display:flex;align-items:stretch;gap:12px;margin-bottom:4px">
    <div style="display:flex;flex-direction:column;align-items:center;width:32px;flex-shrink:0">
      <div style="width:28px;height:28px;border-radius:50%;background:#38bdf822;border:1px solid #38bdf8;
                  display:flex;align-items:center;justify-content:center;
                  font-size:0.7em;font-weight:700;color:#38bdf8">3</div>
      <div style="width:1px;background:#1e293b;flex:1;margin-top:4px"></div>
    </div>
    <div style="background:#0d1420;border:1px solid #38bdf833;border-radius:8px;
                padding:10px 14px;flex:1;margin-bottom:4px">
      <div style="color:#38bdf8;font-size:0.78em;font-weight:700">👤 Load Student Profile</div>
      <div style="color:#64748b;font-size:0.72em;margin-top:2px">name, expertise, style preferences</div>
      <div style="color:#475569;font-size:0.68em;margin-top:4px">← covered in Part 2</div>
    </div>
  </div>

  <!-- step 4 -->
  <div style="display:flex;align-items:stretch;gap:12px;margin-bottom:4px">
    <div style="display:flex;flex-direction:column;align-items:center;width:32px;flex-shrink:0">
      <div style="width:28px;height:28px;border-radius:50%;background:#fbbf2422;border:1px solid #fbbf24;
                  display:flex;align-items:center;justify-content:center;
                  font-size:0.7em;font-weight:700;color:#fbbf24">4</div>
      <div style="width:1px;background:#1e293b;flex:1;margin-top:4px"></div>
    </div>
    <div style="background:#0d1420;border:1px solid #fbbf2433;border-radius:8px;
                padding:10px 14px;flex:1;margin-bottom:4px">
      <div style="color:#fbbf24;font-size:0.78em;font-weight:700">🔄 Detect Profile Changes</div>
      <div style="color:#64748b;font-size:0.72em;margin-top:2px">update expertise if user contradicts profile</div>
      <div style="color:#475569;font-size:0.68em;margin-top:4px">← covered in Part 4</div>
    </div>
  </div>

  <!-- step 5 -->
  <div style="display:flex;align-items:stretch;gap:12px;margin-bottom:4px">
    <div style="display:flex;flex-direction:column;align-items:center;width:32px;flex-shrink:0">
      <div style="width:28px;height:28px;border-radius:50%;background:#34d39922;border:1px solid #34d399;
                  display:flex;align-items:center;justify-content:center;
                  font-size:0.7em;font-weight:700;color:#34d399">5</div>
      <div style="width:1px;background:#1e293b;flex:1;margin-top:4px"></div>
    </div>
    <div style="background:#0d1420;border:1px solid #34d39933;border-radius:8px;
                padding:10px 14px;flex:1;margin-bottom:4px">
      <div style="color:#34d399;font-size:0.78em;font-weight:700">🧩 Assemble Context Window</div>
      <div style="color:#64748b;font-size:0.72em;margin-top:2px">system prompt + history + RAG + new message</div>
      <div style="color:#475569;font-size:0.68em;margin-top:4px">← covered in Part 5 visualizer</div>
    </div>
  </div>

  <!-- step 6 -->
  <div style="display:flex;align-items:stretch;gap:12px;margin-bottom:4px">
    <div style="display:flex;flex-direction:column;align-items:center;width:32px;flex-shrink:0">
      <div style="width:28px;height:28px;border-radius:50%;background:#f9e2af22;border:1px solid #f9e2af;
                  display:flex;align-items:center;justify-content:center;
                  font-size:0.7em;font-weight:700;color:#f9e2af">6</div>
      <div style="width:1px;background:#1e293b;flex:1;margin-top:4px"></div>
    </div>
    <div style="background:#0d1420;border:1px solid #f9e2af33;border-radius:8px;
                padding:10px 14px;flex:1;margin-bottom:4px">
      <div style="color:#f9e2af;font-size:0.78em;font-weight:700">🤖 LLM Inference → Stream Reply</div>
      <div style="color:#64748b;font-size:0.72em;margin-top:2px">generate response token by token</div>
      <div style="color:#475569;font-size:0.68em;margin-top:4px">← production: vLLM + SSE</div>
    </div>
  </div>

  <!-- step 7 -->
  <div style="display:flex;align-items:stretch;gap:12px">
    <div style="display:flex;flex-direction:column;align-items:center;width:32px;flex-shrink:0">
      <div style="width:28px;height:28px;border-radius:50%;background:#94b8d422;border:1px solid #94b8d4;
                  display:flex;align-items:center;justify-content:center;
                  font-size:0.7em;font-weight:700;color:#94b8d4">7</div>
    </div>
    <div style="background:#0d1420;border:1px solid #94b8d433;border-radius:8px;
                padding:10px 14px;flex:1">
      <div style="color:#94b8d4;font-size:0.78em;font-weight:700">💾 Persist Turn + Update Profile</div>
      <div style="color:#64748b;font-size:0.72em;margin-top:2px">save to Redis (recent) + PostgreSQL (long-term)</div>
      <div style="color:#475569;font-size:0.68em;margin-top:4px">← production: Redis + PostgreSQL</div>
    </div>
  </div>

</div>
"""

display(HTML(flow_html))


# 📝 Summary: Zoe's Complete LLM Context Engineering Lab

---

## 🗺️ The Journey, Part by Part

| Part | Prof. Eric's request | Zoe's solution |
|------|---------------------|----------------|
| 1 | "Can it remember our conversation?" | `messages` list — pass full history every call |
| 2 | "Can it adapt to 300 different students?" | Profile dict → auto-generated system prompt |
| 3 | "Why is it getting slower?" | History compression — summarise old turns |
| 3b | *(Zoe's own discovery)* | Anti-patterns: what Zoe almost got wrong |
| 3c | "Can it know about our course materials?" | Simple RAG — inject external knowledge |
| 3d | "Which compression strategy is best?" | Truncation strategies: sliding window vs compression vs extraction |
| 4 | "A student contradicted their earlier profile" | Conflict detection — auto-update profile |
| 5 | "How would this work in production?" | Production architecture map + request lifecycle |

---

## 🔁 The Mental Model

Every student message flows through the same pipeline:
```
Student message
    ↓
[ Safety guard ]              ← blocks injection / PII
    ↓
Retrieve RAG docs             ← Part 3c (course materials)
    ↓
Load student profile          ← Part 2
    ↓
Detect profile changes        ← Part 4
    ↓
Assemble context window       ← Part 3d visualizer
    ↓
LLM inference → stream reply  ← Production: vLLM + SSE
    ↓
Persist turn + update profile ← Production: Redis + PostgreSQL
```

You built every one of those steps in this notebook.

---

> **Context management is the invisible backbone of every LLM product you've ever used.**  
> You now understand what most LLM tutorials skip entirely. That's a real head start. 🎉
